# GND IDs for all corpora

One notebook, one export. Run it top to bottom.

| step | what | fills |
|---|---|---|
| **1** | load all corpora, take the IDs that ship with the metadata (ELTeC, RoKo, DTA, Novellenschatz; EcoCor via Wikidata) | `provided_in_metadata`, `wikidata` |
| **2** | find the missing IDs: **2a** string match against the big GND file, **2b** copy IDs from the same author in other corpora (RoKo preferred), **2c** lobid search | `exact`, `without_titles`, ..., `filled_from_roko`, `filled_from_other_corpus`, `lobid_match` |
| **3** | all decisions and corrections in one place: **3a** manual decisions (`MANUAL_IDS`), **3b** outdated IDs (GND redirects), **3c** verified single fixes (`ID_FIXES`, `ROW_FIXES`, `YEAR_FIXES`) | `manual_override`, `manual_no_id`, `corrected` |
| **4** | add GND info (name) and canonicity scores, once, with the final IDs | - |
| **5** | checks, nothing is changed: progress, conflicts, wrong IDs, review lists | - |
| **6** | export to `resources/final` (the **only** export): one row per work + one row per author | - |

**One row per (work, author).** While the notebook runs, `all_works` has one row for every author of a work (`work_id`, `author_pos`, `n_authors`), so matching, manual IDs, fixes and checks always deal with single persons. Step 4 marks the author with the highest canonicity score of each work (`is_primary`); step 6 uses that author to export **one row per work** again (`data_all.csv`, per-corpus files) and adds `work_authors.csv` with one row per author.

Steps 1-2 only fill IDs that are **still missing**, so a source never overwrites an earlier one.
Step 3 can overwrite anything; `gndIdentifier_orig` keeps the ID from before 3b/3c and `correction` says why something changed.
If a check in step 5 finds a problem, fix it in step 3 and re-run from **Combine everything into one table** (API answers are cached).

# Setup

In [1]:
import json
import re
import time
import unicodedata
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

RES = Path("../resources")
RAW = RES / "raw"
GND_FILE = RES / "gnd_all_mit_namensvarianten.csv"
CANON_FILE = RES / "canonicity_scores.xlsx"
OUT_DIR = RES / "final"                 # step 6 writes here, nothing else is exported
CACHE_DIR = RES / "cache"                  # API answers (Wikidata, lobid) -> re-running does not query again
CACHE_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "gnd-lookup-research/1.0"
PREFERRED_CORPUS = "08_roko"               # step 2b: its ID wins when copying IDs between corpora
SKIP_STATUS = {"multiple_authors", "anonymous"}   # never looked up, never counted as "missing"

EXPORT_COLS = ["corpus", "author_first", "author_last", "filename", "year", "gndIdentifier", "GND_name", "gnd_match_status"]
EXTRA_GND_COLS = []                        # e.g. ["GND_birth", "GND_death"] -> also taken from the GND file in step 4

gnd_all = pd.read_csv(GND_FILE, sep=";", dtype={"gndIdentifier": str})
gnd_all["gndIdentifier"] = gnd_all["gndIdentifier"].str.strip()
gnd_one = gnd_all.drop_duplicates("gndIdentifier").set_index("gndIdentifier")   # one row per ID
print(len(gnd_all), "rows in the GND file,", len(gnd_one), "IDs")

79009 rows in the GND file, 79009 IDs


In [2]:
def load_cache(name, old_path=None):
    """read a JSON cache from CACHE_DIR; falls back to where the old notebooks kept it"""
    for p in [CACHE_DIR / name, old_path]:
        if p is not None and Path(p).exists():
            return json.loads(Path(p).read_text(encoding="utf-8"))
    return {}

def save_cache(name, data):
    (CACHE_DIR / name).write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")

wd_cache = load_cache("wikidata_gnd_cache.json", RES / "01_wikidata" / "wikidata_gnd_cache.json")   # QID -> GND IDs
lobid_cache = load_cache("lobid_cache.json", RES / "02_lobid" / "lobid_cache.json")                 # name search
lobid_id_cache = load_cache("lobid_id_cache.json")                                                  # ID -> record
print(len(wd_cache), "Wikidata,", len(lobid_cache), "lobid searches,", len(lobid_id_cache), "lobid IDs cached")

65 Wikidata, 698 lobid searches, 355 lobid IDs cached


In [3]:
# one ID -> lobid record (cached); used for the names in step 4 and the ID check in step 5e
ID_RE = r"^(\d{6,10}[\dX]|\d{1,8}-[\dX])$"

def lobid_by_id(gid):
    """{'found', 'gndIdentifier', 'preferredName', 'type'} for one ID, or None if lobid could not be reached"""
    if gid in lobid_id_cache:
        return lobid_id_cache[gid]
    for attempt in range(3):
        try:
            r = requests.get(f"https://lobid.org/gnd/{gid}.json", timeout=30, headers={"User-Agent": USER_AGENT})
            if r.status_code in (404, 410):
                rec = {"found": False}
                break
            r.raise_for_status()
            m = r.json()
            rec = {"found": True, "gndIdentifier": m.get("gndIdentifier"),
                   "preferredName": m.get("preferredName"), "type": m.get("type") or []}
            break
        except (requests.RequestException, ValueError):
            time.sleep(2 ** attempt)
    else:
        return None
    lobid_id_cache[gid] = rec
    time.sleep(0.1)
    return rec


## Helpers: names, IDs, progress

In [4]:
def norm(s):
    """casefold, unify unicode, fold umlauts/accents (Müller = Mueller = Muller), drop punctuation (hyphens kept)"""
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKC", str(s)).casefold().replace("ß", "ss")
    s = s.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue")
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    s = re.sub(r"[^\w\s-]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

WRITER_RE = r"Schriftsteller|Autor|Dichter|Erzähler|Lyriker|Dramatiker|Publizist|Übersetzer|Herausgeber|Journalist"
# normalised like the names, so 'Gräfin' -> 'graefin' is recognised as a title
TITLES = {norm(t) for t in ["freiherr", "freiin", "freifrau", "freiherrin", "graf", "gräfin", "baron", "baronin",
                            "ritter", "dr", "prof", "pastor", "fürst", "fürstin", "prinz", "prinzessin"]}
PARTICLES = {"von", "vom", "van", "zu", "zur", "zum", "de", "der", "den", "und"}
SKIP_TOKENS = TITLES | PARTICLES

def drop_tokens(key, words):
    return " ".join(t for t in key.split() if t not in words)

def strip_key(key):
    """remove titles AND particles ('joseph freiherr von eichendorff' -> 'joseph eichendorff')"""
    return drop_tokens(key, SKIP_TOKENS)

def author_key(last, first):
    """one key per author across all corpora: 'raabe, wilhelm'"""
    return last.map(norm) + ", " + first.map(norm)

def clean_id(s):
    """GND IDs as clean strings; '', 'NoID', '118504185.0' etc. handled"""
    s = pd.Series(s, dtype="object").map(lambda v: np.nan if pd.isna(v) else str(v).strip())
    s = s.str.replace(r"\.0$", "", regex=True)
    return s.where(~s.isin(["", "NoID", "nan"]), np.nan)

def person(last, first=np.nan, **extra):
    """one author of a work with several authors: the entries of the column `authors` (see 'Combine everything')"""
    clean = lambda v: np.nan if pd.isna(v) or str(v).strip() == "" else str(v).strip()
    return {"author_last": clean(last), "author_first": clean(first), **extra}

history = {}                                     # missing IDs per corpus after every step (shown in step 5a)

def log_step(label):
    open_ = all_works["gndIdentifier"].isna() & ~all_works["gnd_match_status"].isin(SKIP_STATUS | {"manual_no_id"})
    history[label] = open_.groupby(all_works["corpus"], sort=False).sum()
    print(f"after {label}: {all_works['gndIdentifier'].notna().mean():.1%} of {len(all_works)} author rows have an ID, "
          f"{open_.sum()} still open")

# Step 1: load corpora + IDs from the metadata

Every corpus becomes a table with the same columns:
`author_first, author_last, title, year, filename` and, where the corpus ships one, `gndIdentifier` + `gnd_match_status`.

Works with **several authors** get one more column, `authors`: a list with one entry per person (`person(last, first, ...)`, optionally with that person's own ID). It is split into rows in *Combine everything into one table*. Everything that cannot be split automatically stays one row with the status `multiple_authors` (see `AUTHOR_SPLITS` there).

### 01 Corpus of German-Language Fiction

In [5]:
FILENAME_RE = re.compile(r"^(?P<author>.+?)_-_(?P<title>.+?)(?:_\((?P<year>\d{4})\))?\.txt$")

def parse_txt_filenames(folder):
    """'Heinrich_Laube_-_Eine_Tyroler_Geschichte_(1837).txt' ('(year)' is optional) -> one row per file"""
    rows, unparsed = [], []
    for p in sorted(Path(folder).rglob("*.txt")):
        m = FILENAME_RE.match(p.name)
        if not m:
            unparsed.append(p.name)
            continue
        # last token = last name, rest = first name(s): 'Guy de Maupassant' -> 'Guy de' / 'Maupassant'
        first, _, last = m["author"].replace("_", " ").strip().rpartition(" ")
        rows.append({"author_first": first, "author_last": last,
                     "title": m["title"].replace("_", " ").strip(),
                     "year": float(m["year"]) if m["year"] else np.nan, "filename": p.name})
    if unparsed:
        print(f"{len(unparsed)} files in {folder} did not match the pattern, e.g. {unparsed[:3]}")
    return pd.DataFrame(rows)

fiction_dir = RAW / "01_Corpus of German-Language Fiction"
de_fiction = parse_txt_filenames(fiction_dir / "corpus-of-german-fiction-txt")
translated = parse_txt_filenames(fiction_dir / "corpus-of-translated-foreign-language-fiction-txt")
print(len(de_fiction), "german,", len(translated), "translated")

2735 german, 484 translated


### 03 de-corp (fiction + non-fiction)

In [6]:
de_corp = pd.read_csv(RAW / "03_de-corp_fiction.csv").drop(columns="Unnamed: 0", errors="ignore")
de_corp_nf = pd.read_csv(RAW / "03_de-corp_non_fiction.csv").drop(columns="Unnamed: 0", errors="ignore")

# placeholders: 'Autor, Unbekannter' / 'Autoren, Unbekannte' / 'Autoren, Verschiedene' -> no ID, never looked up
nf_key = author_key(de_corp_nf["author_last"], de_corp_nf["author_first"])
de_corp_nf["gnd_match_status"] = np.where(nf_key.isin(["autor, unbekannter", "autoren, unbekannte"]), "anonymous",
                                          np.where(nf_key.eq("autoren, verschiedene"), "multiple_authors", None))
print(len(de_corp), "fiction,", len(de_corp_nf), "non-fiction")

4215 fiction, 765 non-fiction


### 04 d-prose 1870-1920

Metadata CSV (`04_d-prose_V2_norm_year.csv`, `;`-separated) instead of the folder with the txt files. It ships clean first / last names, the title, the file name (without `.txt`) and the year (`norm_year`, always 4 digits; `verwendetes Datum` is the raw date, sometimes a range like `1914-1922/1967`). No IDs.

As in corpus 10, a particle stays at the end of the first name (`'Ferdinand von'` / `'Saar'`). Two works have two authors (`'Felix Dahn und Therese Dahn'`): they are split into persons via the column `authors`. `Klabund` (`'Klabund / Henschke'` in the CSV) is one person.

In [7]:
LEADING_PARTICLES = {"von", "vom", "van", "zu", "zur", "zum", "de", "der", "den"}   # also used by corpus 10

d_prose_raw = pd.read_csv(RAW / "04_d-prose_V2_norm_year.csv", sep=";", encoding="utf-8-sig", dtype=str)

def particle_to_first(first, last):
    """'Marie' / 'von Ebner-Eschenbach' -> 'Ebner-Eschenbach' / 'Marie von' (same convention as the RoKo / 19C names)"""
    words = last.split()
    n = 0
    while n < len(words) - 1 and words[n].lower() in LEADING_PARTICLES:
        n += 1
    return " ".join(words[n:]), f"{first} {' '.join(words[:n])}".strip()

def d_prose_names(first, last, full):
    """one (last, first, status, authors) per row of the CSV"""
    first, last, full = first.strip(), last.strip(), full.strip()
    if re.search(r"\bund\b", first):                        # 'Felix und Therese' / 'Dahn und Dahn' -> take the persons from the full name
        people = []
        for part in re.split(r"\s+und\s+", full):          # 'Felix Dahn und Therese Dahn'
            f, _, l = part.rpartition(" ")
            l, f = particle_to_first(f, l)
            people.append(person(l, f))
        return full, np.nan, "multiple_authors", people
    if "/" in last:                                          # 'Klabund / Henschke': pseudonym + real name -> one person, the name used in the corpus
        return full, np.nan, None, np.nan
    last, first = particle_to_first(first, last)
    return last, first or np.nan, None, np.nan

parsed = [d_prose_names(f, l, g) for f, l, g in zip(d_prose_raw["Vorname Autor"], d_prose_raw["Nachname Autor"],
                                                    d_prose_raw["Gesamtname Autor"])]
d_prose = pd.DataFrame({
    "author_last": [p[0] for p in parsed],
    "author_first": [p[1] for p in parsed],
    "title": d_prose_raw["Titel"].str.strip(),
    "year": pd.to_numeric(d_prose_raw["norm_year"], errors="coerce"),      # normalised first year ('verwendetes Datum' is the raw date)
    "filename": d_prose_raw["Dateiname"],                                  # without '.txt'
    "gnd_match_status": [p[2] for p in parsed],
    "authors": pd.Series([p[3] for p in parsed], index=d_prose_raw.index, dtype=object),
})
assert d_prose["filename"].is_unique
print(len(d_prose), "works,", d_prose["year"].isna().sum(), "without a year,", d_prose["gnd_match_status"].eq("multiple_authors").sum(),
      "with several authors")

2511 works, 0 without a year, 2 with several authors


### 05 ELTeC-deu (ships GND IDs)

In [8]:
eltec_raw = pd.read_csv(RAW / "05_ELTeC-deu_metadata.tsv", sep="\t")
name_parts = eltec_raw["author-name"].str.strip().replace("", np.nan).str.partition(",")   # 'Arnim, Bettina von'

eltec = pd.DataFrame({
    "author_last": name_parts[0].str.strip(),
    "author_first": name_parts[2].str.strip(),
    "title": eltec_raw["title"],
    "year": eltec_raw["reference-year"],
    "filename": eltec_raw["filename"],
    # 'http://d-nb.info/gnd/118504185' (also the odd '.../gnd/pnd:118535455') -> '118504185'
    "gndIdentifier": eltec_raw["author-ids"].str.extract(r"([0-9]+[0-9X])$")[0],
})
eltec["gnd_match_status"] = np.where(eltec["gndIdentifier"].notna(), "provided_in_metadata", None)

# blank author names: take them from the GND file via the ID
blank = eltec["author_last"].isna() & eltec["gndIdentifier"].notna()
gnd_parts = eltec.loc[blank, "gndIdentifier"].map(gnd_one["GND_name"]).str.partition(",")
eltec.loc[blank, "author_last"] = gnd_parts[0].str.strip()
eltec.loc[blank, "author_first"] = gnd_parts[2].str.strip().replace("", np.nan)
print(len(eltec), "works,", eltec["gndIdentifier"].notna().sum(), "with ID from the metadata")

100 works, 100 with ID from the metadata


### 06 Romane der Romantik

In [9]:
# 'RoRo Gesamtkorpus' already contains the other three sheets (Kernkorpus, Ergänzung I + II)
roro_raw = pd.read_excel(RAW / "06_Romane" / "Romane der Romantik.xlsx", sheet_name="RoRo Gesamtkorpus")
name_parts = roro_raw["Autor:in"].str.strip().str.partition(",")      # 'Novalis' / 'Jean Paul' have no comma
roro = pd.DataFrame({
    "author_last": name_parts[0].str.strip(),
    "author_first": name_parts[2].str.strip().replace("", np.nan),
    "title": roro_raw["Titel"],
    "year": roro_raw["Datum Erstdruck"],
    "filename": roro_raw["Dateiname"],
})
print(len(roro), "works,", "filenames unique" if roro["filename"].is_unique else "DUPLICATE filenames")

115 works, filenames unique


### 07 Bildungsromankorpus

In [10]:
bild_raw = pd.read_csv(RAW / "07_korpus-bildungsromane-main" / "Bildungsromankorpus_Metadaten.csv",
                       sep=";", encoding="utf-8-sig")
authors = bild_raw["Autor*in"].str.strip()
multi = authors.str.contains(",", na=False)          # 'Georg Heizen, Uwe Koch' -> several authors in one cell
name_parts = authors.str.rpartition(" ")             # 'Hugo Ball' -> 'Hugo' / 'Ball'
bild = pd.DataFrame({
    "author_first": name_parts[0].replace("", np.nan).where(~multi, np.nan),
    "author_last": name_parts[2].where(~multi, authors),
    "title": bild_raw["Titel"],
    "year": bild_raw["Erstveröffentlichung, Deutsch"],   # first German publication
    "filename": bild_raw["Dateiname"],                   # empty where there is no full text
    "gnd_match_status": np.where(multi, "multiple_authors", None),
})
def split_cell(cell):
    """'Georg Heizen, Uwe Koch' -> two persons ('Georg' / 'Heizen', 'Uwe' / 'Koch')"""
    people = []
    for part in cell.split(","):
        first, _, last = part.strip().rpartition(" ")
        if last:
            people.append(person(last, first))
    return people if len(people) > 1 else np.nan

bild["authors"] = pd.Series([split_cell(a) if m else np.nan for a, m in zip(authors, multi)], index=bild.index, dtype=object)
print(len(bild), "works,", bild["filename"].isna().sum(), "without full text,", multi.sum(), "with several authors")

220 works, 94 without full text, 2 with several authors


### 08 RoKo (ships GND IDs)

In [11]:
roko_raw = pd.read_csv(RAW / "08_metadata.csv", dtype={"author_gnd_id": str})
name_parts = roko_raw["author"].str.strip().str.partition(",")        # 'Alexis, Willibald'; 'Novalis'
roko = pd.DataFrame({
    "author_last": name_parts[0].str.strip(),
    # 'Ompteda, Georg, Freiherr von' -> 'Georg Freiherr von'
    "author_first": name_parts[2].str.replace(",", "", regex=False).str.strip().replace("", np.nan),
    "title": roko_raw["title"],
    "year": roko_raw["year"],
    "filename": roko_raw["id"],                       # corpus ID ('filename_orig' = name in the source repo)
    "gndIdentifier": roko_raw["author_gnd_id"].str.strip(),
})
roko["gnd_match_status"] = np.where(roko["gndIdentifier"].notna(), "provided_in_metadata", None)
print(len(roko), "works,", roko["gndIdentifier"].notna().sum(), "with ID from the metadata")

1834 works, 1833 with ID from the metadata


### 09 EcoCor (ships Wikidata IDs -> GND via Wikidata P227)

In [12]:
TEI = {"tei": "http://www.tei-c.org/ns/1.0"}

def _text(el):
    return re.sub(r"\s+", " ", "".join(el.itertext())).strip() if el is not None else ""

def _year(el):
    if el is None:
        return np.nan
    m = re.search(r"\d{4}", el.get("when") or _text(el))
    return float(m.group()) if m else np.nan

def parse_ecocor(folder):
    rows = []
    for p in sorted(Path(folder).glob("*.xml")):
        fd = ET.parse(p).getroot().find("tei:teiHeader/tei:fileDesc", TEI)
        authors = fd.findall("tei:titleStmt/tei:author", TEI)
        # (name, QID) per author, so that every QID stays with its author
        named = [(re.sub(r"\s*\([^)]*\)\s*$", "", _text(a)),                                   # drop '(1730-1788)'
                  m.group(1) if (m := re.search(r"(Q\d+)\s*$", a.get("ref") or "")) else np.nan)
                 for a in authors if _text(a)]
        names = [n for n, _ in named]
        qids = [q for _, q in named if pd.notna(q)]
        last, _, first = (names[0] if names else "").partition(",")
        year = _year(fd.find("tei:sourceDesc/tei:bibl[@type='firstEdition']/tei:date", TEI))
        if np.isnan(year):
            year = _year(fd.find("tei:sourceDesc/tei:bibl[@type='printSource']/tei:date", TEI))
        if np.isnan(year) and (m := re.match(r"(\d{4})_", p.name)):          # '1754_Gessner_Daphnis.xml'
            year = float(m.group(1))
        rows.append({
            # several authors -> 'A / B' (the string match reports multiple_authors)
            "author_last": " / ".join(names) if len(names) > 1 else (last.strip() or np.nan),
            "author_first": np.nan if len(names) > 1 else (first.strip() or np.nan),
            "title": _text(fd.find("tei:titleStmt/tei:title", TEI)),
            "year": year,
            "filename": p.name,
            "wikidata_id": qids[0] if len(names) == 1 and len(qids) == 1 else np.nan,
            # several authors: one entry per person (split into rows in 'Combine everything')
            "authors": [person(n.partition(",")[0], n.partition(",")[2], wikidata_id=q) for n, q in named]
                       if len(named) > 1 else np.nan,
        })
    return pd.DataFrame(rows)

eco = parse_ecocor(RAW / "09_eco-de-main" / "tei")
print(len(eco), "works,", eco["wikidata_id"].notna().sum(), "with a Wikidata ID")

190 works, 121 with a Wikidata ID


In [13]:
WD_URL = "https://query.wikidata.org/sparql"

def wikidata_to_gnd(qids, batch=200):
    """QID -> sorted list of GND IDs (P227). QIDs that could not be queried are not cached (retried next run)."""
    todo = sorted(set(qids) - set(wd_cache))
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        query = ("SELECT ?item ?gnd WHERE { VALUES ?item { %s } OPTIONAL { ?item wdt:P227 ?gnd } }"
                 % " ".join(f"wd:{q}" for q in chunk))
        for attempt in range(3):
            try:
                r = requests.post(WD_URL, data={"query": query}, timeout=60,
                                  headers={"User-Agent": USER_AGENT, "Accept": "application/sparql-results+json"})
                r.raise_for_status()
                bindings = r.json()["results"]["bindings"]
                break
            except (requests.RequestException, ValueError, KeyError):
                time.sleep(2 ** attempt)
        else:
            print(f"Wikidata not reachable for {len(chunk)} QIDs -> run the cell again")
            continue
        found = {q: set() for q in chunk}
        for b in bindings:
            qid = b["item"]["value"].rsplit("/", 1)[-1]
            if "gnd" in b:
                found.setdefault(qid, set()).add(b["gnd"]["value"].strip())
        wd_cache.update({q: sorted(v) for q, v in found.items()})
        save_cache("wikidata_gnd_cache.json", wd_cache)
        time.sleep(1)
    return {q: wd_cache[q] for q in qids if q in wd_cache}

def pick_gnd(ids):
    """(gndIdentifier or None, status) from the GND IDs of one Wikidata item"""
    if not ids:
        return None, None
    if len(ids) == 1:
        return ids[0], "wikidata"
    in_file = [i for i in ids if i in gnd_one.index]          # several P227 values: keep the one in our GND file
    if len(in_file) == 1:
        return in_file[0], "wikidata+disambiguated"
    return None, "wikidata_ambiguous"                         # kept as a hint for step 3a

multi_qids = {p["wikidata_id"] for lst in eco["authors"].dropna() for p in lst if pd.notna(p["wikidata_id"])}
wd_gnd = wikidata_to_gnd(sorted(set(eco["wikidata_id"].dropna()) | multi_qids))
picked = eco["wikidata_id"].map(lambda q: pick_gnd(wd_gnd.get(q, [])) if pd.notna(q) else (None, None))
eco["gndIdentifier"] = picked.map(lambda t: t[0])
eco["gnd_match_status"] = picked.map(lambda t: t[1])
for lst in eco["authors"].dropna():            # works with several authors: every author has their own QID
    for p in lst:
        p["gndIdentifier"], p["gnd_match_status"] = (pick_gnd(wd_gnd.get(p["wikidata_id"], []))
                                                     if pd.notna(p["wikidata_id"]) else (None, None))
print(eco["gnd_match_status"].value_counts(dropna=False))

gnd_match_status
wikidata              120
NaN                    69
wikidata_ambiguous      1
Name: count, dtype: int64


### 10 19th-century prose corpus (Kanon_Status = original canonicity)

In [14]:
prose_raw = pd.read_csv(RAW / "10_19CproseCorpus.tsv", sep="\t", dtype=str)

INITIALS_RE = r"(?:[^\W\d_]{1,3}\.\s*)+"              # 'v.W.', 'O. L.', 'Bn.' -> initials only, no lookup
PROSE_PLACEHOLDERS = {"o n", "unbekannt"}             # norm('o.N.') = 'o n'

def prose_names(last, first):
    """one (last, first, status) per row of the TSV"""
    last = "" if pd.isna(last) else re.sub(r"\s*\([^)]*\)", "", last).strip()     # 'Dmitrijew (erschienen o.N.)'
    first = "" if pd.isna(first) else first.strip()
    first = re.split(r"d\.\s*i\.\s*:?", first)[-1].strip()                        # 'Hans Warring, d.i.: Emma' -> 'Emma'
    if re.search(r"vollendet|\bund\b", first) or re.search(r",|/|&", last):
        return f"{last}, {first}".strip(", "), np.nan, "multiple_authors"       # 'Heimburg / E. Marlitt, vollendet von ...'
    first = re.sub(r"\s*,\s*", " ", first).strip()                               # 'Lina, Freifrau von' -> 'Lina Freifrau von'
    if (not last and not first) or norm(last) in PROSE_PLACEHOLDERS or (not last and norm(first) in PROSE_PLACEHOLDERS):
        return last or np.nan, first or np.nan, "anonymous"
    if not first and " " in last:                       # full name in 'Nachname': 'Arnold Kasten'
        first, _, last = last.rpartition(" ")
    elif not last:                                      # full name in 'Vorname': 'Fritz Kalle' / 'Dorothea'
        first, _, last = first.rpartition(" ")
    words = last.split()                                # 'von Perin' -> 'Perin' / 'Josephine von'
    n = 0
    while n < len(words) - 1 and words[n].lower() in LEADING_PARTICLES:
        n += 1
    if n:
        last, first = " ".join(words[n:]), f"{first} {' '.join(words[:n])}".strip()
    status = "anonymous" if re.fullmatch(INITIALS_RE, f"{first} {last}".strip()) else None
    return last, first or np.nan, status

parsed = [prose_names(l, f) for l, f in zip(prose_raw["Nachname"], prose_raw["Vorname"])]
prose = pd.DataFrame({
    "author_last": [p[0] for p in parsed],
    "author_first": [p[1] for p in parsed],
    "title": prose_raw["Titel"].str.strip(),
    "year": pd.to_numeric(prose_raw["Jahr_ED"].str.extract(r"(\d{4})")[0], errors="coerce"),
    "filename": prose_raw["Dokument_ID"],
    "gnd_match_status": [p[2] for p in parsed],
})
# last name only ('Goethe', 'heinse'): take the first name from the same corpus if exactly one author has that last name
full = prose.dropna(subset=["author_first"])
full = full[full["gnd_match_status"].isna()].assign(k=lambda d: d["author_last"].map(norm))
by_last = full.groupby("k").agg(n=("author_first", "nunique"), last=("author_last", "first"), first=("author_first", "first"))
by_last = by_last[by_last["n"] == 1]
only_last = prose["author_first"].isna() & prose["gnd_match_status"].isna() & prose["author_last"].map(norm).isin(by_last.index)
k = prose.loc[only_last, "author_last"].map(norm)
prose.loc[only_last, "author_first"] = k.map(by_last["first"])
prose.loc[only_last, "author_last"] = k.map(by_last["last"])
print(len(prose), "works,", prose["gnd_match_status"].value_counts().to_dict(), "|", only_last.sum(), "first names filled in")

5277 works, {'anonymous': 201, 'multiple_authors': 56} | 23 first names filled in


### 11 DTA-Kernkorpus Belletristik (ships GND IDs in the TEI header)

In [15]:
GND_REF_RE = re.compile(r"d-nb\.info/gnd/(?:pnd:)?(\d+[\dX]|\d+-[\dX])")
DTA_ANONYMOUS = {"anonym", "anonymus", "n n", "unbekannt"}

def read_tei_header(path):
    """only the teiHeader (the files are large, the text is not needed)"""
    for _, el in ET.iterparse(path, events=("end",)):
        if el.tag == "{http://www.tei-c.org/ns/1.0}teiHeader":
            return el
    return None

def parse_dta(folder):
    rows = []
    for p in sorted(Path(folder).rglob("*.xml")):
        header = read_tei_header(p)
        fd = header.find("tei:fileDesc", TEI)
        names, ids = [], []
        for a in fd.findall("tei:titleStmt/tei:author", TEI):
            pn = a.find("tei:persName", TEI)
            pn = a if pn is None else pn
            last, first = _text(pn.find("tei:surname", TEI)), _text(pn.find("tei:forename", TEI))
            if not last:                                 # name without <surname>: 'Novalis', 'Anonym'
                last = _text(pn)
            if last:
                names.append((last, first))
                m = GND_REF_RE.search(pn.get("ref") or a.get("ref") or "")
                ids.append(m.group(1) if m else None)
        year = _year(fd.find("tei:sourceDesc/tei:biblFull/tei:publicationStmt/tei:date[@type='publication']", TEI))
        if np.isnan(year) and (m := re.search(r"_(\d{4})$", p.name.split(".")[0])):   # 'lasswitz_seife_1890.TEI-P5.xml'
            year = float(m.group(1))
        multi = len(names) > 1
        anon = not names or norm(names[0][0]) in DTA_ANONYMOUS
        # several authors: one entry per person, each with the ID of its own <author> (split into rows in 'Combine everything')
        people = [person(l, f, gndIdentifier=i,
                         gnd_match_status="anonymous" if norm(l) in DTA_ANONYMOUS else "provided_in_metadata" if i else None)
                  for (l, f), i in zip(names, ids)] if multi else np.nan
        rows.append({
            "author_last": " / ".join(f"{l}, {f}".strip(", ") for l, f in names) if multi else (names[0][0] if names else np.nan),
            "author_first": np.nan if multi or not names else (names[0][1] or np.nan),
            "title": _text(fd.find("tei:titleStmt/tei:title[@type='main']", TEI)) or _text(fd.find("tei:titleStmt/tei:title", TEI)),
            "subtitle": _text(fd.find("tei:titleStmt/tei:title[@type='sub']", TEI)) or np.nan,
            "year": year,
            "filename": p.name,
            "authors": people,
            "gndIdentifier": ids[0] if len(names) == 1 and not anon else np.nan,
            "gnd_match_status": "multiple_authors" if multi else "anonymous" if anon else
                                ("provided_in_metadata" if ids[0] else None),
        })
    return pd.DataFrame(rows)

dta = parse_dta(RAW / "11_dta_kernkorpus_belletristik_2026-02-10")
print(len(dta), "works,", dta["gndIdentifier"].notna().sum(), "with ID from the metadata")

553 works, 534 with ID from the metadata


### 12 Disko classics

A few author names in the CSV are misspelled or incomplete. `DISKO_NAME_FIXES` corrects them **for the lookup only**
(the export keeps the original `Autor` column).

In [16]:
disko_raw = pd.read_csv(RAW / "12_diskoclassics.csv", dtype=str)

DISKO_NAME_FIXES = {                       # name as in the CSV -> 'Last, First' used for matching
    "Peledan": "Péladan, Joséphin",
    "Foster, E.M.": "Forster, E. M.",
    "Meyring, Gustav": "Meyrink, Gustav",
    "Wilkies, Collins": "Collins, Wilkie",
    "Broch, Herrmann": "Broch, Hermann",
    "Scherbart, Paul": "Scheerbart, Paul",
    "Beermann, Richard A.": "Bermann, Richard A.",
    "Goedsche (Retcliffe, John Sir)": "Goedsche, Hermann",
    "Sheridan LeFanu, Joseph": "Le Fanu, Joseph Sheridan",
    "Casanova": "Casanova, Giacomo",
    "Dauthendey": "Dauthendey, Max",
    "Alcott": "Alcott, Louisa May",
    "d’Aurevilly": "Barbey d'Aurevilly, Jules",
    "d’Aurevilly, Jules Amédée Barbey": "Barbey d'Aurevilly, Jules",
    "Marquis de Sade": "Sade, Donatien Alphonse François de",
}

def disko_names(author):
    a = DISKO_NAME_FIXES.get(author.strip(), author.strip())
    a = re.sub(r"\s*\([^)]*\)", "", a).strip()          # 'Wilde, Oscar (mutmaßlich)'
    if norm(a) in {"anonym", "anonymus", "unbekannt"}:
        return a, np.nan, "anonymous", np.nan
    if "&" in a:                                        # 'Brecht, Bertolt & Feuchtwanger, Lion'
        people = [person(p.partition(",")[0], p.partition(",")[2]) for p in a.split("&") if p.strip()]
        return a, np.nan, "multiple_authors", people if len(people) > 1 else np.nan
    last, _, first = a.partition(",")
    return last.strip(), first.strip() or np.nan, None, np.nan

parsed = [disko_names(a) for a in disko_raw["Autor"]]
disko = pd.DataFrame({
    "author_last": [p[0] for p in parsed],
    "author_first": [p[1] for p in parsed],
    "title": disko_raw["Titel"],
    "year": pd.to_numeric(disko_raw["Jahr"].str.extract(r"(\d{4})")[0], errors="coerce"),   # '1919, 1921 & 1931' -> 1919
    "filename": np.nan,                                 # the CSV has no file names
    "gnd_match_status": [p[2] for p in parsed],
    "authors": pd.Series([p[3] for p in parsed], dtype=object),
})
print(len(disko), "works,", disko["gnd_match_status"].value_counts().to_dict())

86 works, {'anonymous': 2, 'multiple_authors': 1}


### 13 Deutscher Novellenschatz (ships GND IDs in the TEI header)

One TEI file per novella (`schmid_mohrenfranzl_1910.xml`), same header layout as the DTA files in corpus 11, so `parse_dta` is reused as it is.
`year` is the year of the digitised edition (`publicationStmt/date`, e.g. 1910 = 2nd edition), not necessarily the first publication.

In [17]:
novellenschatz = parse_dta(RAW / "13_deutscher_novellenschatz")
print(len(novellenschatz), "works,", novellenschatz["gndIdentifier"].notna().sum(), "with ID from the metadata,",
      novellenschatz["gnd_match_status"].value_counts(dropna=False).to_dict())

87 works, 87 with ID from the metadata, {'provided_in_metadata': 87}


### 14 Neuer Deutscher Novellenschatz

One row per novella: `surname` / `name` already split, first publication year and place, volume, life dates and an `alias` column
(pseudonyms, birth names). No IDs and no file names. As in corpus 10, a particle stays at the end of the first name (`'Ida von'`).
The CSV is `;`-separated and has hundreds of empty trailing columns (dropped here); the original columns are kept in the per-corpus export (step 6).

In [18]:
neuer_raw = pd.read_csv(RAW / "14_neuer_deutscher_novellenschatz.csv", sep=";", encoding="utf-8-sig")
neuer_raw = neuer_raw.loc[:, ~neuer_raw.columns.str.startswith("Unnamed")]       # empty trailing columns

neuer = pd.DataFrame({
    "author_last": neuer_raw["surname"].str.strip().replace("", np.nan),
    "author_first": neuer_raw["name"].str.strip().replace("", np.nan),         # 'Ida ' -> 'Ida'
    "title": neuer_raw["title"].str.strip(),
    "year": neuer_raw["publ.year"],
    "filename": np.nan,                                 # the CSV has no file names
})
print(len(neuer), "works,", neuer["year"].isna().sum(), "without a year")

70 works, 9 without a year


## Combine everything into one table

`all_works` has **one row per (work, author)**: a work with one author is one row, a work with several authors is one row per author.
So every step below (name matching, manual IDs, fixes, checks) deals with single persons.

| column | meaning |
|---|---|
| `work_id` | `corpus:src_row`, identifies the work (`filename` can be empty) |
| `author_pos`, `n_authors` | position of the author within the work (1, 2, ...) and number of authors |
| `author_role` | `author` (default); `completer`, `editor`, ... if given in `AUTHOR_SPLITS` |

Several authors are split **automatically** where the metadata lists them (07 Bildungsromane, 09 EcoCor, 11 DTA, 13 Novellenschatz,
12 Disko classics: column `authors`) and **by hand** with `AUTHOR_SPLITS` (all other corpora, e.g. 10 19C-prose).
Placeholders stay one row as they are (`multiple_authors` e.g. 'Autoren, Verschiedene', `anonymous`).
Step 4 marks one author per work (`is_primary` = the highest canonicity score), step 6 exports one row per work with that author.

To split a work by hand: add it to `AUTHOR_SPLITS` and re-run from here.


In [19]:
AUTHOR_SPLITS = {
    # works with several authors that the corpus parsers could not split.
    # key:   the name as it stands in the table, written like in MANUAL_IDS ('First Last'; no comma needed),
    #        or (name, corpus) to limit it to one corpus
    # value: one (last, first, role) per person, in the order they should get as author_pos
    #        role: "author" / "completer" / "editor" / ... (only informational)
    # the list under the table below shows the works that are still unsplit
    "Theodor von / Cornelius, Wilhelm Kobbe" : [("Cornelius", "Wilhelm", "author"), ("Kobbe", "Theodor von", "author")],
    "Ferdinand/Schücking, Levin Freiligrath" : [("Schücking", "Levin", "author"), ("Freiligrath", "Ferdinand", "author")],
    "Alfred/Taschenberg, Ernst Ludwig Brehm" : [("Taschenberg", "Ernst Ludwig", "author"), ("Brehm", "Alfred", "author")],
    "Johanna/Kinkel, Gottfried Kinkel" : [("Kinkel", "Johanna", "author"), ("Kinkel", "Gottfried", "author")],
    "Wackenroder, Tieck" : [("Tieck", "Ludwig", "author"), ("Wackenroder", "Wilhelm Heinrich", "author")],
    "Grimm, Jacob und Wilhelm" : [("Grimm", "Jacob", "author"), ("Grimm", "Wilhelm", "author")],
    "Brüder Grimm" : [("Grimm", "Jacob", "author"), ("Grimm", "Wilhelm", "author")],
    "Felix / Dahn, Therese Dahn" : [("Dahn", "Felix", "author"), ("Dahn", "Therese", "author")],
    "Apel/Laun" : [("Apel", "August", "editor"), ("Laun", "Friedrich", "editor")],
    "Gisela und Bettina von Arnim": [("Arnim", "Bettina", "author"), ("Arnim", "Gisela", "author")],
    "Heimburg E. Marlitt vollendet von Wilhelmine": [("Marlitt", "E.", "author"), ("Heimburg", "Wilhelmine", "completer")],
}

SPLIT_LOOKUP = {}
for who, people in AUTHOR_SPLITS.items():
    name, corpus = who if isinstance(who, tuple) else (who, None)
    SPLIT_LOOKUP[(norm(name), corpus)] = [person(l, f, author_role=r) for l, f, r in people]


def expand_authors(d, corpus):
    """step-1 table of one corpus (one row per work) -> one row per (work, author).
    Works with one author, and placeholders that cannot be split, stay one row."""
    d = d.reset_index(drop=True)
    rows = []
    for src_row, rec in enumerate(d.to_dict("records")):
        people = rec.pop("authors", None)                         # list of persons from the corpus parser (or NaN)
        if not isinstance(people, list):                          # not split by the parser: AUTHOR_SPLITS?
            key = (norm(rec.get("author_first")) + " " + norm(rec.get("author_last"))).strip()
            people = SPLIT_LOOKUP.get((key, corpus)) or SPLIT_LOOKUP.get((key, None))
        people = people or [None]
        for pos, p in enumerate(people, 1):
            row = dict(rec, corpus=corpus, src_row=src_row, work_id=f"{corpus}:{src_row}",
                       author_pos=pos, n_authors=len(people), author_role="author")
            if p is not None:
                gid = p.get("gndIdentifier", np.nan)
                row.update(author_last=p["author_last"], author_first=p["author_first"], gndIdentifier=gid,
                           wikidata_id=p.get("wikidata_id", np.nan), author_role=p.get("author_role", "author"),
                           gnd_match_status=p.get("gnd_match_status") or ("provided_in_metadata" if pd.notna(gid) else None))
            rows.append(row)
    return pd.DataFrame(rows)


In [20]:
corpora = {
    "01_german-fiction": de_fiction,
    "01_translated-fiction": translated,
    "03_de-corp": de_corp,
    "03_de-corp-non-fiction": de_corp_nf,
    "04_d-prose": d_prose,
    "05_ELTeC-deu": eltec,
    "06_romane-der-romantik": roro,
    "07_bildungsromane": bild,
    "08_roko": roko,
    "09_ecocor": eco,
    "10_19C-prose": prose,
    "11_dta-belletristik": dta,
    "12_diskoclassics": disko,
    "13_deutscher-novellenschatz": novellenschatz,
    "14_neuer-deutscher-novellenschatz": neuer,
}
# src_row = position of the work in its corpus table (step 6 uses it to re-attach the original metadata columns)
STD_COLS = ["author_first", "author_last", "title", "year", "filename", "gndIdentifier", "gnd_match_status", "wikidata_id"]
AUTHOR_COLS = ["work_id", "author_pos", "n_authors", "author_role"]     # see the table above

all_works = pd.concat([expand_authors(d, name).reindex(columns=[*STD_COLS, "corpus", "src_row", *AUTHOR_COLS])
                       for name, d in corpora.items()], ignore_index=True)
all_works["year"] = pd.to_numeric(all_works["year"], errors="coerce")
all_works[["src_row", "author_pos", "n_authors"]] = all_works[["src_row", "author_pos", "n_authors"]].astype(int)
all_works["gndIdentifier"] = clean_id(all_works["gndIdentifier"])
all_works["gnd_match_status"] = all_works["gnd_match_status"].astype(object).where(all_works["gnd_match_status"].notna(), np.nan)
all_works["author_key"] = author_key(all_works["author_last"], all_works["author_first"])

n_multi = all_works.loc[all_works["n_authors"] > 1, "work_id"].nunique()
print(f"{len(all_works)} author rows for {all_works['work_id'].nunique()} works "
      f"({n_multi} works with several authors) in {len(corpora)} corpora")
log_step("1 metadata")
all_works.groupby("corpus", sort=False)["gnd_match_status"].value_counts().unstack(fill_value=0)

19335 author rows for 19242 works (93 works with several authors) in 15 corpora
after 1 metadata: 14.0% of 19335 author rows have an ID, 16421 still open


gnd_match_status,anonymous,multiple_authors,provided_in_metadata,wikidata,wikidata_ambiguous
corpus,,,,,
08_roko,0,0,1833,0,0
11_dta-belletristik,4,0,564,0,0
10_19C-prose,201,0,0,0,0
09_ecocor,0,0,0,120,1
05_ELTeC-deu,0,0,100,0,0
13_deutscher-novellenschatz,0,0,87,0,0
03_de-corp-non-fiction,2,1,0,0,0
12_diskoclassics,2,0,0,0,0


In [21]:
# works with several authors: check the split
split_rows = all_works[all_works["n_authors"] > 1]
print(split_rows["work_id"].nunique(), "works were split into authors")
display(split_rows[["corpus", "title", "author_pos", "author_role", "author_first", "author_last", "gndIdentifier", "gnd_match_status"]])

# still ONE row although the metadata names several authors -> split them by hand in AUTHOR_SPLITS above
# (placeholders like 'Autoren, Verschiedene' and anonymous works are meant to stay as they are)
unsplit = all_works[all_works["gnd_match_status"].eq("multiple_authors") & all_works["n_authors"].eq(1)
                    & all_works["author_key"].ne("autoren, verschiedene")]
print(len(unsplit), "works with several authors are still one row; copy-paste template for AUTHOR_SPLITS:\n")
for t in unsplit.drop_duplicates(["corpus", "author_last"]).itertuples():
    print(f'    "{t.author_last}": [("Last", "First", "author"), ("Last", "First", "author")],   # {t.corpus}: {t.title}')


93 works were split into authors


,corpus,title,author_pos,author_role,author_first,author_last,gndIdentifier,gnd_match_status
3276,03_de-corp,Gespensterbuch. Erstes Bändchen,1,editor,August,Apel,NaN,NaN
3277,03_de-corp,Gespensterbuch. Erstes Bändchen,2,editor,Friedrich,Laun,NaN,NaN
3278,03_de-corp,Gespensterbuch. Viertes Bändchen,1,editor,August,Apel,NaN,NaN
3279,03_de-corp,Gespensterbuch. Viertes Bändchen,2,editor,Friedrich,Laun,NaN,NaN
3280,03_de-corp,Gespensterbuch. Drittes Bändchen,1,editor,August,Apel,NaN,NaN
...,...,...,...,...,...,...,...,...
19023,11_dta-belletristik,Gedichte,2,author,Friedrich Leopold zu,Stolberg-Stolberg,118755552,provided_in_metadata
19053,11_dta-belletristik,Herzensergießungen eines kunstliebenden Kloste...,1,author,Wilhelm Heinrich,Wackenroder,118628097,provided_in_metadata
19054,11_dta-belletristik,Herzensergießungen eines kunstliebenden Kloste...,2,author,Ludwig,Tieck,12989432X,provided_in_metadata
19096,12_diskoclassics,Leben Eduards des Zweiten von England,1,author,Bertolt,Brecht,NaN,NaN


0 works with several authors are still one row; copy-paste template for AUTHOR_SPLITS:



# Step 2: find the missing IDs

Three sources, each one only fills IDs that are **still missing**: **2a** string match, **2b** same author in other corpora, **2c** lobid search.

## 2a String match with the big GND file

Name keys from strict to loose (full name -> without titles -> without particles -> one given name),
homonyms are disambiguated with the year and the occupation. Runs per corpus (earliest year per corpus, as before).

In [22]:
def build_gnd_dicts(gnd):
    """(gnd, raw_dict, stripped_dict): name key -> sorted list of row positions"""
    gnd = gnd.reset_index(drop=True)
    raw, stripped = defaultdict(set), defaultdict(set)

    def add(i, name):
        k = norm(name)
        if k:
            raw[k].add(i)
            stripped[strip_key(k)].add(i)

    for col in ["GND_name_ohne_komma", "GND_name_ein_vorname", "GND_name_ohne_titel"]:
        for i, name in enumerate(gnd[col]):
            add(i, name)
    for i, name in enumerate(gnd["GND_name"]):          # 'Raabe, Wilhelm' -> 'Wilhelm Raabe'
        if isinstance(name, str) and "," in name:
            last, first = [p.strip() for p in name.split(",", 1)]
            add(i, f"{first} {last}")
        else:
            add(i, name)
    fin = lambda d: {k: sorted(v) for k, v in d.items()}
    return gnd, fin(raw), fin(stripped)


def _pick(cand, year):
    """disambiguate homonyms with the year and the occupation; returns a (possibly still >1) frame"""
    if len(cand) > 1 and year is not None and pd.notna(year):
        ok = cand[cand.GND_birth.isna() | (cand.GND_birth <= year - 10)]        # born before the work
        cand = ok if not ok.empty else cand
        alive = cand[(cand.GND_death.isna() | (cand.GND_death + 5 >= year))     # lifespan covers year
                     & (cand.GND_birth.isna() | (cand.GND_birth <= year))]      # (+5: posthumous prints)
        cand = alive if not alive.empty else cand
    if len(cand) > 1:
        w = cand[cand.GND_occupation.str.contains(WRITER_RE, na=False)]
        cand = w if not w.empty else cand
    if len(cand) > 1:
        n = cand[["GND_birth", "GND_death", "placeOfBirth", "GND_occupation"]].notna().sum(axis=1)
        cand = cand[n == n.max()]
    return cand


def get_gnd_identifier(first, last, year=None, *, gnd, raw_dict, stripped_dict):
    """(gndIdentifier or None, status), trying progressively looser name keys"""
    first = "" if pd.isna(first) else first
    if re.search(r"\bund\b|/|&|\bbrüder\b", f"{first} {last}", flags=re.I):
        return None, "multiple_authors"
    full = norm(f"{first} {last}")
    first_given = norm(first).split(" ")[0] if norm(first) else ""
    tiers = [
        ("exact",             raw_dict,      full),
        ("without_titles",    raw_dict,      drop_tokens(full, TITLES)),
        ("without_particles", stripped_dict, strip_key(full)),
        ("first_given_name",  stripped_dict, strip_key(norm(f"{first_given} {last}"))),
    ]
    for g in strip_key(norm(first)).split()[1:]:       # 'Christian Friedrich Hebbel' -> 'Friedrich Hebbel'
        tiers.append(("other_given_name", stripped_dict, strip_key(norm(f"{g} {last}"))))
    seen = set()
    for tier, d, key in tiers:
        if not key or (id(d), key) in seen:
            continue
        seen.add((id(d), key))
        idx = d.get(key)
        if not idx:
            continue
        cand = _pick(gnd.iloc[idx], year)
        if len(cand) == 1:
            return cand.iloc[0].gndIdentifier, tier + ("" if len(idx) == 1 else "+disambiguated")
        return None, "ambiguous"
    return None, "no_match"


GND_INDEX = build_gnd_dicts(gnd_all)                  # built once (slow part)

def stringmatch(df):
    """name lookup for the rows of df; each author once, with their earliest year.
    Returns stringmatch_id / stringmatch_status with the same index as df."""
    gnd, raw, stripped = GND_INDEX
    d = df[["author_first", "author_last", "year"]].copy()
    d["author_first"] = d["author_first"].fillna("")
    d.index.name = "_row"
    authors = d.groupby(["author_last", "author_first"])["year"].min().reset_index()
    res = [get_gnd_identifier(a.author_first, a.author_last, a.year, gnd=gnd, raw_dict=raw, stripped_dict=stripped)
           for a in authors.itertuples(index=False)]
    authors["stringmatch_id"] = [r[0] for r in res]
    authors["stringmatch_status"] = [r[1] for r in res]
    out = (d.reset_index()
             .merge(authors.drop(columns="year"), on=["author_last", "author_first"], how="left")
             .set_index("_row"))
    return out[["stringmatch_id", "stringmatch_status"]]

In [23]:
# looked up for EVERY row (also those with an ID from the metadata) -> step 5 compares the two
all_works["stringmatch_id"] = pd.Series(np.nan, index=all_works.index, dtype=object)
all_works["stringmatch_status"] = pd.Series(np.nan, index=all_works.index, dtype=object)
lookup = all_works["author_last"].notna() & ~all_works["gnd_match_status"].isin(SKIP_STATUS)
for corpus, sub in all_works[lookup].groupby("corpus", sort=False):
    res = stringmatch(sub)
    all_works.loc[res.index, "stringmatch_id"] = res["stringmatch_id"]
    all_works.loc[res.index, "stringmatch_status"] = res["stringmatch_status"]

# fill only what is still missing
fill = all_works["gndIdentifier"].isna() & all_works["stringmatch_id"].notna() & lookup
all_works.loc[fill, "gndIdentifier"] = all_works.loc[fill, "stringmatch_id"]
all_works.loc[fill, "gnd_match_status"] = all_works.loc[fill, "stringmatch_status"]
# still missing: note why (no_match / ambiguous / multiple_authors), but keep an earlier hint like wikidata_ambiguous
why = all_works["gndIdentifier"].isna() & all_works["gnd_match_status"].isna()
all_works.loc[why, "gnd_match_status"] = all_works.loc[why, "stringmatch_status"]

print(fill.sum(), "works filled by the string match")
log_step("2a string match")
all_works.loc[fill, "gnd_match_status"].value_counts()

13783 works filled by the string match
after 2a string match: 85.3% of 19335 author rows have an ID, 2637 still open


gnd_match_status
exact                             12443
exact+disambiguated                 835
first_given_name                    153
without_particles                   138
without_titles                      115
other_given_name                     73
other_given_name+disambiguated       16
first_given_name+disambiguated       10
Name: count, dtype: int64

## 2b Same author in other corpora

If the author has exactly one ID in `08_roko`, that ID is used. Otherwise the ID is copied if all corpora agree on one.
IDs that are already there are never overwritten.

In [24]:
has_id = all_works.dropna(subset=["gndIdentifier"])
ids_all = has_id.groupby("author_key")["gndIdentifier"].agg(lambda s: sorted(s.unique()))
ids_pref = (has_id[has_id["corpus"] == PREFERRED_CORPUS]
            .groupby("author_key")["gndIdentifier"].agg(lambda s: sorted(s.unique())))
pref_id = ids_pref[ids_pref.str.len() == 1].str[0]
agreed_id = ids_all[ids_all.str.len() == 1].str[0]
fill_id = pref_id.combine_first(agreed_id)

fill = (all_works["gndIdentifier"].isna() & all_works["author_key"].isin(fill_id.index)
        & ~all_works["gnd_match_status"].isin(SKIP_STATUS))
from_pref = all_works["author_key"].isin(pref_id.index)
all_works.loc[fill, "gndIdentifier"] = all_works.loc[fill, "author_key"].map(fill_id)
all_works.loc[fill, "gnd_match_status"] = np.where(from_pref[fill], "filled_from_roko", "filled_from_other_corpus")

print(fill.sum(), "works filled,", all_works.loc[fill, "author_key"].nunique(), "authors")
log_step("2b other corpora")

132 works filled, 28 authors
after 2b other corpora: 86.0% of 19335 author rows have an ID, 2505 still open


## 2c Lobid search for the authors that still have no ID

Only authors with **no ID in any corpus** (authors with conflicting IDs are a step 3a decision).
Every hit is scored:

- **name**: full name / first given name / first initial must fit `preferredName` or a `variantName` (titles and particles ignored)
- **years**: dropped if born less than 10 years before the earliest work; lifespan covering the works scores higher
- **occupation**: small bonus for writer/author/poet etc.

`accepted` = clearly the best hit (full or first-given-name match, >= 2 points ahead of the runner-up) -> filled.
`review` / `none` -> step 3a.

In [25]:
LOBID_URL = "https://lobid.org/gnd/search"
KEEP = ["gndIdentifier", "preferredName", "variantName", "dateOfBirth", "dateOfDeath", "professionOrOccupation", "type"]

def lobid_search(q, size=50):
    """persons matching q (slim records), or None if lobid could not be reached"""
    key = f"{q}|{size}"
    if key in lobid_cache:
        return lobid_cache[key]
    params = {"q": q, "filter": "type:Person", "format": "json", "size": size}
    for attempt in range(3):
        try:
            r = requests.get(LOBID_URL, params=params, timeout=30, headers={"User-Agent": USER_AGENT})
            r.raise_for_status()
            members = r.json().get("member", [])
            break
        except (requests.RequestException, ValueError):
            time.sleep(2 ** attempt)
    else:
        return None
    lobid_cache[key] = [{k: m.get(k) for k in KEEP} for m in members]
    time.sleep(0.1)
    return lobid_cache[key]

def toks(s):
    return [t for t in norm(s).split() if t not in SKIP_TOKENS]

def clean_for_query(s):
    if pd.isna(s):
        return ""
    return " ".join(t for t in re.sub(r"[^\w\s-]", " ", str(s)).split() if norm(t) not in SKIP_TOKENS)

def query_tiers(first, last):
    """(query, size) from strict to loose"""
    f, l = clean_for_query(first), clean_for_query(last)
    if not l:
        return []
    given = f.split()
    tiers = []
    if f:
        tiers.append((f"{f} {l}", 50))
    if len(given) > 1:
        tiers.append((f"{given[0]} {l}", 50))
    if f and len(l) >= 4:
        tiers.append((l, 100))          # last name only (first name still has to fit when scoring)
    if not f:
        tiers.append((l, 50))           # pseudonyms like 'Novalis'
    return tiers

def parse_year(v):
    if isinstance(v, list):
        v = v[0] if v else None
    m = re.search(r"\d{4}", str(v)) if v else None
    return int(m.group()) if m else None

def name_score(c, first_t, last_t):
    """3 = same full name, 2 = last name + first/any given name, 1 = same first initial, 0 = no match"""
    tf, tl = toks(first_t), toks(last_t)
    if not tl:
        return 0
    best = 0
    for form in [f for f in [c.get("preferredName"), *(c.get("variantName") or [])] if f]:
        if "," in form:
            l, _, f = form.partition(",")
            cl, cf = toks(l), toks(f)
        else:
            t = toks(form)
            cl, cf = (tl, t[:-len(tl)]) if t[-len(tl):] == tl else (t, [])
        if cl != tl:
            continue
        if cf == tf:
            s = 3
        elif cf and tf and cf[0] == tf[0]:
            s = 2
        elif tf and tf[0] in cf:                       # Rufname
            s = 2
        elif cf and tf and cf[0][0] == tf[0][0]:
            s = 1
        else:
            s = 0
        best = max(best, s)
    return best

def year_score(c, ymin, ymax):
    """(score, birth, death); score None = cannot have written the work"""
    b, d = parse_year(c.get("dateOfBirth")), parse_year(c.get("dateOfDeath"))
    if pd.isna(ymin):
        return 0, b, d
    ymin = int(ymin)
    ymax = int(ymax) if pd.notna(ymax) else ymin
    if b is not None and b > ymin - 10:
        return None, b, d
    s = 0
    if b is not None:
        s += 1
        if d is None and b < ymin - 100:
            s -= 3
    if d is not None:
        if d >= ymax:
            s += 3
        elif d >= ymin:
            s += 2
        elif ymin - d <= 30:
            s -= 1
        else:
            s -= 3
    elif b is not None:
        s += 2
    return s, b, d

def score_candidates(members, first, last, ymin, ymax):
    out = []
    for c in members:
        ns = name_score(c, first, last)
        if ns == 0:
            continue
        ys, b, d = year_score(c, ymin, ymax)
        if ys is None:
            continue
        occ = [o.get("label") for o in (c.get("professionOrOccupation") or []) if isinstance(o, dict) and o.get("label")]
        total = 2 * ns + ys + (1 if re.search(WRITER_RE, " ".join(occ)) else 0)
        out.append({"gndIdentifier": c["gndIdentifier"], "name": c.get("preferredName"), "birth": b, "death": d,
                    "occupation": "; ".join(occ[:3]), "name_score": ns, "score": total})
    return sorted(out, key=lambda r: -r["score"])

def lookup_author(first, last, ymin, ymax):
    tiers = query_tiers(first, last)
    if not tiers:
        return {"status": "none", "n_candidates": 0}
    scored, used = [], None
    for q, size in tiers:
        members = lobid_search(q, size)
        if members is None:
            return {"status": "error", "query": q}
        used = q
        scored = score_candidates(members, first, last, ymin, ymax)
        if scored:
            break
    if not scored:
        return {"status": "none", "query": used, "n_candidates": 0}
    top, second = scored[0], (scored[1] if len(scored) > 1 else None)
    ok = top["name_score"] >= 2 and (second is None or top["score"] - second["score"] >= 2)
    return {"status": "accepted" if ok else "review", "query": used, "n_candidates": len(scored),
            "gndIdentifier": top["gndIdentifier"], "lobid_name": top["name"], "birth": top["birth"],
            "death": top["death"], "occupation": top["occupation"], "score": top["score"],
            "runner_up_id": second["gndIdentifier"] if second else None,
            "runner_up_name": second["name"] if second else None,
            "runner_up_years": f"{second['birth']}-{second['death']}" if second else None}

In [26]:
PLACEHOLDERS = {"anonymous", "anonymus", "anonym", "unbekannt", "autor", "autoren"}

n_ids = all_works.groupby("author_key")["gndIdentifier"].nunique()
todo_mask = (all_works["author_key"].isin(n_ids[n_ids == 0].index)
             & ~all_works["gnd_match_status"].isin(SKIP_STATUS)
             & all_works["author_last"].notna()
             & ~all_works["author_last"].map(norm).isin(PLACEHOLDERS))
todo = (all_works[todo_mask].groupby("author_key")
        .agg(author_last=("author_last", "first"), author_first=("author_first", "first"),
             n_works=("corpus", "size"), year_min=("year", "min"), year_max=("year", "max"))
        .reset_index().sort_values("n_works", ascending=False))
print(len(todo), "authors to look up")

rows = []
for i, a in enumerate(todo.itertuples(index=False), 1):
    res = lookup_author(a.author_first, a.author_last, a.year_min, a.year_max)
    rows.append({"author_key": a.author_key, "author_first": a.author_first, "author_last": a.author_last,
                 "n_works": a.n_works, "year_min": a.year_min, "year_max": a.year_max, **res})
    if i % 25 == 0:
        print(f"{i}/{len(todo)}")
        save_cache("lobid_cache.json", lobid_cache)
save_cache("lobid_cache.json", lobid_cache)

RESULT_COLS = ["author_key", "author_first", "author_last", "n_works", "year_min", "year_max", "status", "query",
               "n_candidates", "gndIdentifier", "lobid_name", "birth", "death", "occupation", "score",
               "runner_up_id", "runner_up_name", "runner_up_years"]
lobid_results = pd.DataFrame(rows).reindex(columns=RESULT_COLS)
if lobid_results["status"].eq("error").any():
    print("lobid was not reachable for some authors -> run this cell again (finished queries are cached)")
lobid_results["status"].value_counts()

578 authors to look up
25/578
50/578
75/578
100/578
125/578
150/578
175/578
200/578
225/578
250/578
275/578
300/578
325/578
350/578
375/578
400/578
425/578
450/578
475/578
500/578
525/578
550/578
575/578


status
accepted    396
review      125
none         57
Name: count, dtype: int64

In [27]:
accepted = lobid_results[lobid_results["status"] == "accepted"].set_index("author_key")
fill = all_works["gndIdentifier"].isna() & all_works["author_key"].isin(accepted.index) & todo_mask
all_works.loc[fill, "gndIdentifier"] = all_works.loc[fill, "author_key"].map(accepted["gndIdentifier"])
all_works.loc[fill, "gnd_match_status"] = "lobid_match"

print(fill.sum(), "works filled for", all_works.loc[fill, "author_key"].nunique(), "authors")
log_step("2c lobid")

# please skim: name, years, occupation
accepted[["author_first", "author_last", "year_min", "year_max", "gndIdentifier", "lobid_name",
          "birth", "death", "occupation", "n_candidates", "n_works"]].sort_values("n_works", ascending=False)

1865 works filled for 396 authors
after 2c lobid: 95.6% of 19335 author rows have an ID, 640 still open


,author_first,author_last,year_min,year_max,gndIdentifier,lobid_name,birth,death,occupation,n_candidates,n_works
author_key,,,,,,,,,,,
"heimburg, w",W.,Heimburg,1880.0,1898.0,104352787,"Heimburg, Wilhelmine",1848.0,1912.0,Schriftstellerin,2,231
"buerstenbinder, elisabeth",Elisabeth,Bürstenbinder,1870.0,1899.0,115341307,"Werner, E.",1838.0,1918.0,Schriftstellerin; Schriftstellerin,1,90
"schmid, herman",Herman,Schmid,1861.0,1877.0,100801919,"Schmid, Hermann von",1815.0,1880.0,Dramatiker; Theaterdirektor; Schriftsteller,12,87
"balzac, honore de",Honorè de,Balzac,1830.0,1834.0,118506358,"Balzac, Honoré de",1799.0,1850.0,Schriftsteller; Verleger,2,81
"godin, a",A.,Godin,1854.0,1887.0,115341080,"Linz, Amélie",1824.0,1904.0,Schriftstellerin,6,53
...,...,...,...,...,...,...,...,...,...,...,...
"sock, paul",Paul,Sock,1921.0,1921.0,131727086X,"Sock, Paul",NaN,NaN,Übersetzer,1,1
"siemens, hans",Hans,Siemens,1923.0,1923.0,1035155125,"Siemens, Hans Peter",1879.0,1961.0,Heimatkundler; Schulleiter,8,1
"semper, karl",Karl,Semper,1873.0,1873.0,119006715,"Semper, Karl",1832.0,1893.0,Zoologe; Naturwissenschaftler; Forschungsreise...,2,1


# Step 3: decisions and corrections

Everything that is decided by hand or verified one by one lives here, in this order:

| part | what |
|---|---|
| **3a** | manual decisions (`MANUAL_IDS`) and the list of authors that are still open |
| **3b** | outdated GND IDs (the GND merged two records, the old ID redirects to the new one) |
| **3c** | verified single fixes (`ID_FIXES`, `ROW_FIXES`, `YEAR_FIXES`) |

## 3a Manual decisions

Write your decisions into `MANUAL_IDS`, then run the cell. Manual entries win over everything else.

- `"Last, First"` or `"First Last"` -> all works of this author in all corpora
- `("Last, First", "05_ELTeC-deu")` -> only this corpus (two different people with the same name)
- value `"NoID"` (or `None`) -> checked, there is no GND record (removes an ID if one was set)

Authors of works with several authors are separate rows: decide for each person.

Below the cell: the list of authors that are **still open**, with the lobid candidates, ready to copy.

In [28]:
MANUAL_IDS = {
    "Christen, Ada": "119549263",
    "Giseke, Robert": "11664818X",
    "Grunert, Carl": "116900733",
    "Karwath, Juliane": "116117729",
    "Raabe, Wilhelm": "118597442",
    "Wassermann, Jakob": "118629387",
    "Wallace, Edgar": "118628739",
    "Scott, Walter": "118612409",
    "Dickens, Charles": "118525239",
    "Grimmelshausen": "118542273",
    "Robert Louis Stevenson": "118753711",
    "FW Hackländer": "118699962",
    "FM Dostojewski": "118527053",
    "Conrad, Joseph": "118521861",
    "Tolstoi, Leo": "11864291X",
    "Werner, Elisabeth": "115341307",
    "H. Clauren": "118521101",
    "Fritz Reck-Mallaczewen": "118598821",
    "Jacob Burckhardt": "11851752X",
    "Elfrid von Taura": "116108118",
    "Jenny Dirnböck-Schulz": "116138645",
    "Ludwig Külz": "133976440",
    "Gustav Hildebrand": "119288087",
    "Alois Theodor Sonnleitner": "119410885",   # 04_d-prose: first names now come from the CSV
    "Ernst Klein": "141494964",
    "Johann Georg Friedrich Jacobi": "102499764",
    "Wilhemine von Hillern": "118904779",
    "Georg Schweinfurth": "118760041",
    "Friedrich Ratzel": "118598538",
    "Karl Gottlieb Samuel Heun": "118521101",
    "Georg Heim": "118709623",
    "Marie Tihanyi Gräfin Sturza": "1348330074",
    "Jakob Frey": "131405330",
    "Alfred Wegener": "118629913",
    "Joseph Baumann": "143499882",
    "Adolf Ebert": "116331992",
    "Paul Weise": "1132540658",
    "Pauline Weise": "NoID",
    "Herman Siebeck": "1019259795",
    "Heinrich Pröhle": "116292199",
    "Lewis Wallace": "11926708X",
    "Willi Seidel": "119037742",
    "J. Walden": "NoID",
    "Johannes Werner": "129421189",
    "Doris Freiin von Spättgen": "117215139",
    "Max Wilke": "NoID",
    "JJA Pffyfer zu Neueck": "124267718",
    "Eduard Vogel": "118627406",
    "Mathias Voglers": "NoID",
    "H[ans] Sturm": "117363839",
    "Klabund": "118562681",   # 04_d-prose: the CSV has one clean author (the old file names gave Moreau / Il / ... as first names)
    "Moritz Schreber": "118610678",
    "Carl Tanera": "117199850",
    "P. Sutter": "NoID",
    "Otto Strauss": "1051176174",
    "Fritz Stüber-Gunther": "129282189",   # 04_d-prose (CSV: Stüber-Gunther, Fritz)
    "Heinrich Schliemann": "118608215",
    "Georg Strelisker": "NoID",
    "Maria von Thilo": "NoID",
    "Walther Rheiner": "11874805X",
    "Max Richter": "NoID",
    "C. E. Ries": "116544635",
    "Marie Tihanyi Sturza": "1348330074",   # 04_d-prose (CSV: Tihanyi Sturza, Marie)
    "Herrmann Ruf": "116767340",
    "Georg Schwarz": "173675700",
    "Jeanne Schultz": "NoID",
    "J. Alois Meier": "1043068686",
    "Ottoma Enking": "116505567",
    "Marie Eugenie delle Grazie": "118746278",   # 04_d-prose (CSV: delle Grazie, Marie Eugenie)
    "Max Dauthenday": "118678973",
    "Princess of Schleswig-Holstein-Sonderburg-Augustenburg Feodora": "1018855998",
    "Karl Geiger": "118813161",
    "Joseph Görres": "11854019X",
    "I. Glässer": "NoID",
    "Reinhard Gerling": "139013156",
    "Adolph von Gerhardt": "1160184763",
    "Edna Fern": "108955544X",
    "Ludwig Friedlaender": "118693433",
    "Martin Fraenkel": "1057568953",
    "Wilhelm Fischer aus Wermelskirchen": "116583959",
    "Charles de Coster": "116583959",           # same ID as Wilhelm Fischer above -> please check (corrected in `ROW_FIXES`, 3c)
    "Otto Berndt": "116144076",
    "Clara Berg": "NoID",
    "Fritz Barschdorff": "NoID",
    "Johann-Conrad Appenzeller": "100796486",
    "Otto von Bismarck": "11851136X",
    "Albert E. Bull": "NoID",
    "Eugen Brunfaut": "NoID",
    "Wilhelm Bolin": "119057832",
    "Joseph von Görres": "11854019X",
    "Oscar Koch": "NoID",
    "Joachim Knappe": "1042647259",
    "K. Labacher": "NoID",
    "Maria Ludolfs": "115475761",
    "Margareta Sophia Liebeskind": "100804195",
    "Rodolf Hawel": "137795106",
    "Gustav (Johann Friedrich Gustav) Harders": "NoID",
    "Julius Hahn": "1018778209",
    "Bodo Keith": "NoID",
    "Heinrich Hackmann": "119511835",
    "Julius Guttmann": "122512324",
    "Caroline von Günderrode": "118543202",
    "Hans Grimm": "118542249",
    "Francis Grillparzer": "118542192",
    "Ferdinand Gribble": "NoID",
    "Max Henning": "116716533",
    "Johann Rudolf Wyß der Jüngere": "117349666",
    "C. Hoffmann": "NoID",
    "Wilhelm Hertz": "119405334",
    "Hermann Fürst von Pückler": "118597019",
    "E. Werber": "NoID",   # lobid: 117295485 Werber, Elise
    "Arnold Kasten": "NoID",   # lobid: 127476814 Kasten, August
    "Hans i.e. Babette von Arnold Bülow": "116363207",
    "v. Karl Heigel": "116650818",
    "M. Corvus": "NoID",
    "Theodor Karl Schultz": "117202045",
    "C. Lauckner": "116858370",   # lobid: 116858370 Sudermann, Clara
    "H. v. Götzendorff-Grabowski": "127680381",   # lobid: 127680381 Götzendorff-Grabowski, Helene von
    "A. Weber": "NoID",   # lobid: 1156197538 Weber, Gustavus A.
    "H. [Adele Wild Wesemael]": "11730834X",
    "Ludwig Rosen": "117221627",   # lobid: 101720726 Rosen, Gottlieb von
    "Anonymous": "NoID",
    "R. Godin": "NoID",   # lobid: 1099914000 Godin, Richard
    "Etmar Weidrod": "NoID",
    "Anna von Peternell": "NoID",   # lobid: 135935229 Peternell, August
    "Louise Ernesti": "117062049",   # lobid: 102446660 Ernesti, Heinrich Friedrich Theodor Ludwig
    "August Karl Heigel": "116650818",
    "August von Karl Heigel": "116650818",
    "M. v. Dorsner": "NoID",
    "H. Schmid": "100801919",   # lobid: 1313642908 Schmid, Hugo
    "N. A. i.e.: August Justus Gutmann Mordtmann": "126853517",
    "F. Meister": "104428921X",   # lobid: 104428921X Meister, Friedrich
    "K. Horstemann": "NoID",
    "Wilhelm Kästner": "NoID",   # lobid: 188440038 Kästner, Wilhelm
    "E. v. Wang": "NoID",   # lobid: 171903978 Wang, Neng E.
    "A. Lichtenstern": "NoID",   # lobid: 1328489280 Lichtenstern, Alexander
    "J. E. Mand i.e.: Friedrich August von Karl Mecklenburg-Strelitz": "120815893",
    "Fr. von Bülow": "11706646X",   # lobid: 11706646X Bülow, Frieda von
    "H. Pichler": "11617790X",   # lobid: 105738528X Pichler, Richard H.
    "L. Otto": "1069206210",   # lobid: 1069206210 Otto, Luise
    "Emil Karl Franzos": "118702599",
    "Wilhelm Berger": "121847268",   # lobid: 121847268 Berger, Wilhelm
    "Emil Roland": "116969431",   # lobid: 116969431 Lewald, Emmi
    "C. F. Meyer": "118581775",   # lobid: 118581775 Meyer, Conrad Ferdinand
    "C. Sch…z": "NoID",
    "Henriette von Schrorn": "116922842",
    "Gustav Meyrinck": "118582046",   # lobid: 118582046 Meyrink, Gustav
    "Anna Weitzer": "NoID",
    "Ludwig Neusser": "NoID",   # lobid: 116985186 Neusser, Lambert
    "Frank Norris": "118588702",   # lobid: 118588702 Norris, Frank
    "H. v. Osten": "1045211427",   # lobid: 1045211427 Osten, Hans von
    "V. Weingarten": "NoID",
    "bernguseck": "NoID",
    "Alfred Pförtner": "NoID",   # lobid: 1141813262 Pförtner, Andreas
    "Franz Wallner": "118806025",   # lobid: 118806025 Wallner, Franz
    "Wilhelm von Pochhammer": "172310326",   # lobid: 172310326 Martell, Wilhelm
    "W. Usener": "117322199",   # lobid: 117322199 Usener, Wilhelm
    "Wilhelm von Pochhanner": "172310326",
    "Wenzel Treml": "116891025",
    "Gebriele Reuter": "118744712",   # lobid: 1218987286 Reuter, Gisbert
    "Theodore": "NoID",   # lobid: 1055161090 Theodore
    "B. Cressieux": "NoID",
    "Karl Larsen": "116745789",   # lobid: 116745789 Larsen, Karl
    "meiselhess": "119017660",
    "Ivan Iwanowitsch Dmitrijew": "117645729",   # lobid: 117645729 Dmitrijew, Iwan Iwanowitsch
    "Dorothea": "NoID",   # lobid: 135202868 Dorothea
    "Hans Eichelbach": "118531026",
    "Elise": "NoID",   # lobid: 1069336157 Elise
    "Engel": "NoID",   # lobid: 1254182535 Engel
    "Felder": "NoID",
    "I. Frapan": "1063133041",   # lobid: 1063133041 Frapan, Ilse
    "Friederike": "NoID",
    "Goedsche": "118599879",
    "Edith Gräfin Salzburg": "116566795",   # lobid: 10246653X Erhardus, de Salzburg
    "R. Hartmann": "NoID",   # lobid: 1034745042 Rosenbusch, Karl Heinrich Wilhelm Louis
    "heinse": "118548565",
    "Gustav von Herringen": "116565616",
    "Hofmannsthal": "118552759",
    "D. Friedrich Jacobs": "117039225",   # lobid: 117039179 Jacobs, Dorothea
    "Amelie Bölte": "116223715",
    "Sophi Bernhardi-Tieck": "118973665",   # lobid: 118973665 Bernhardi, Sophie
    "Frau von Krudener": "119160145",   # lobid: 136535666 Krüdener, Frederik von
    "Lafontaine": "NoID",   # lobid: 1243937793 LaFontaine
    "Godard D’Aucourt": "100207669",
    "Henrich Laube": "118570080",   # lobid: 130217263 Laube, Hermann
    "Bernhardi-Tieck": "118973665",
    "Lichtenstein": "NoID",   # lobid: 126829721 Lichtenstein
    "Aline": "NoID",
    "kon," : "NoID",
    "kvk," : "NoID",
    ("Roth, Joseph", "10_19C-prose"): "118603140",
    ("Musäus, Johann Karl August", "11_dta-belletristik") : "118735365",
    ("Frölich, Henriette", "11_dta-belletristik") : "131967630",
    "Johhann Wolfgang Goethe" : "118540238",
    "Goethe" : "118540238",
    "Ritter, Johann Wilhelm" : "118745468",
    ("Arnim", "10_19C-prose"): "118504177",
    ("Heun", "10_19C-prose"): "118521101",
    ("Wackenroder", "10_19C-prose"): "118628097",
    ("Vulpius", "10_19C-prose"): "118770489",
    "Fock" : "118691988",
    "Guntram" : "119501201",
    "Armand" : "118756044",
    "rein" : "117374997",
    "Gottlob Adolf Ernst von Nostitz und Jänkendorf" : "117058955",
    "Lohmann" : "118182765",
    "J. Sch-r" : "NoID",
    ("Marlitt, E.", "10_19C-prose"): "118731092",   # the work 'Heimburg E. Marlitt vollendet von Wilhelmine' is split into two authors (AUTHOR_SPLITS)
    ("Otto Müller", "14_neuer-deutscher-novellenschatz"): "117609145",
    "Margarethe von Wolff" : "117441708",
    "Adolf Pichler" : "118742353",
    "Laurids Kruse" : "116575301",
    "August Apel": "116315865",   # lobid: 116315865 Apel, Johann August
    "Wilhelm Cornelius": "1341917541",   # lobid: 1341917541 Cornelius, Wilhelm
    "Malwine Enkhausen": "116471778",
    "Georg Heizen": "109942647",
    "Friederike Lohmann" : "117196126",
    "Heinrich Zschokke" : "118637266",
    "Achim von Arnim" : "118504177",
    ("Carl Grosse", "08_roko"): "120671883",
    # ("Wassermann, Jakob", "05_ELTeC-deu"): "118629387",
}

NO_ID_VALUES = {"NoID", "noid", ""}

def manual_mask(who):
    author, corpus = who if isinstance(who, tuple) else (who, None)
    last, _, first = author.partition(",")
    mask = all_works["author_key"] == norm(last) + ", " + norm(first)
    if "," not in author:                                  # 'Edgar Wallace' (First Last) works as well
        first_last = (all_works["author_first"].map(norm) + " " + all_works["author_last"].map(norm)).str.strip()
        mask |= first_last == norm(author)
    if corpus is not None:
        mask &= all_works["corpus"] == corpus
    return mask

def apply_manual_ids(overrides):
    not_found = []
    for who, gid in overrides.items():
        mask = manual_mask(who)
        if not mask.any():
            not_found.append(who)
            continue
        if gid is None or str(gid).strip() in NO_ID_VALUES:
            all_works.loc[mask, "gndIdentifier"] = np.nan
            all_works.loc[mask, "gnd_match_status"] = "manual_no_id"
            continue
        new_id = str(gid).strip()
        differs = mask & (all_works["gndIdentifier"].fillna("") != new_id)    # only these change status
        all_works.loc[mask, "gndIdentifier"] = new_id
        all_works.loc[differs, "gnd_match_status"] = "manual_override"
    print(len(overrides) - len(not_found), "of", len(overrides), "manual entries applied")
    if not_found:
        print("WARNING - no works found for (typo in name or corpus?):", not_found)

apply_manual_ids(MANUAL_IDS)
log_step("3a manual")

220 of 220 manual entries applied
after 3a manual: 98.1% of 19335 author rows have an ID, 0 still open


In [29]:
all_works.query("author_last=='Grosse'")

,author_first,author_last,title,year,filename,gndIdentifier,gnd_match_status,wikidata_id,corpus,src_row,work_id,author_pos,n_authors,author_role,author_key,stringmatch_id,stringmatch_status
4396,Julius,Grosse,Ravensbeck,1910.0,Julius_Grosse_-_Ravensbeck.txt,118718983,exact,NaN,03_de-corp,1167,03_de-corp:1167,1,1,author,"grosse, julius",118718983,exact
4397,Karl,Grosse,Der Genius,1791.0,Karl_Grosse_-_Der_Genius.txt,116869852,exact,NaN,03_de-corp,1168,03_de-corp:1168,1,1,author,"grosse, karl",116869852,exact
12181,Carl,Grosse,Ravensbeck,1910.0,Grosse_Ravensbeck,120671883,manual_override,NaN,08_roko,1015,08_roko:1015,1,1,author,"grosse, carl",120671883,exact
12182,Carl,Grosse,Der Genius,1791.0,Grosse_Genius,120671883,manual_override,NaN,08_roko,1016,08_roko:1016,1,1,author,"grosse, carl",120671883,exact
13838,Julius,Grosse,Vetter Isidor,1874.0,00420-00,118718983,exact,NaN,10_19C-prose,632,10_19C-prose:632,1,1,author,"grosse, julius",118718983,exact
19200,Julius,Grosse,Vetter Isidor,1910.0,grosse_isidor_1910.xml,118718983,provided_in_metadata,NaN,13_deutscher-novellenschatz,22,13_deutscher-novellenschatz:22,1,1,author,"grosse, julius",118718983,exact


In [30]:
# all_works.query("title=='Das Eulenhaus'")

In [31]:
# still open: no ID, not skipped, not decided manually -> add them to MANUAL_IDS above and run from there
still_open = all_works["gndIdentifier"].isna() & ~all_works["gnd_match_status"].isin(SKIP_STATUS | {"manual_no_id"})
open_authors = (all_works[still_open].groupby("author_key")
                .agg(author_first=("author_first", "first"), author_last=("author_last", "first"),
                     n_works=("corpus", "size"), corpora=("corpus", lambda s: ", ".join(sorted(s.unique()))),
                     year_min=("year", "min"), year_max=("year", "max"),
                     status=("gnd_match_status", "first"))
                .reset_index()
                .merge(lobid_results[["author_key", "status", "gndIdentifier", "lobid_name", "birth", "death",
                                      "runner_up_id", "runner_up_name", "runner_up_years"]]
                       .rename(columns={"status": "lobid_status", "gndIdentifier": "lobid_best_id"}),
                       on="author_key", how="left")
                .sort_values("n_works", ascending=False))
print(len(open_authors), "authors still open, copy-paste template:\n")
for a in open_authors.itertuples():
    name = f"{'' if pd.isna(a.author_first) else a.author_first} {a.author_last}".strip()
    hint = f"   # lobid: {a.lobid_best_id} {a.lobid_name}" if pd.notna(a.lobid_best_id) else ""
    print(f'    "{name}": "XXX",{hint}')
open_authors

0 authors still open, copy-paste template:



,author_key,author_first,author_last,n_works,corpora,year_min,year_max,status,lobid_status,lobid_best_id,lobid_name,birth,death,runner_up_id,runner_up_name,runner_up_years


## 3b Outdated GND IDs

When the GND merges two records of one person, the old ID redirects to the new one (HTTP 301).
Checked are all IDs of the canon list **and** of the corpora that are not in the big GND file. Answers are cached
(`cache/gnd_redirect_cache.json`), a second run does not ask again.

From here on `all_works` is changed by corrections: `gndIdentifier_orig` keeps the ID as 3a left it, `correction` says what was changed and why.

In [32]:
redirect_cache = load_cache("gnd_redirect_cache.json")

def check_id(gid):
    """{'status': 200|301|404|..., 'target': new ID for a 301}, or None if lobid could not be reached"""
    if gid in redirect_cache:
        return redirect_cache[gid]
    for attempt in range(3):
        try:
            r = requests.get(f"https://lobid.org/gnd/{gid}.json", allow_redirects=False, timeout=30,
                             headers={"User-Agent": USER_AGENT})
            rec = {"status": r.status_code}
            if r.status_code in (301, 302):
                m = re.search(r"/gnd/([0-9X-]+)", r.headers.get("Location", ""))
                rec["target"] = m.group(1) if m else None
            break
        except requests.RequestException:
            time.sleep(2 ** attempt)
    else:
        return None
    redirect_cache[gid] = rec
    time.sleep(0.1)
    return rec

def resolve(gid):
    """follow a chain of redirects"""
    seen = set()
    while gid in redirect_cache and redirect_cache[gid].get("target") and gid not in seen:
        seen.add(gid)
        gid = redirect_cache[gid]["target"]
    return gid

# the canon list as shipped (step 4 attaches its scores via the resolved IDs)
canon_list = pd.read_excel(CANON_FILE, dtype={"GND": str}).drop(columns="Unnamed: 0", errors="ignore")
canon_list["GND"] = clean_id(canon_list["GND"])

# 3b and 3c change all_works in place -> they must run once per fresh table (see the note at the top)
assert "gndIdentifier_orig" not in all_works, "3b/3c already ran on this table: re-run from 'Combine everything into one table'"
all_works["gndIdentifier_orig"] = all_works["gndIdentifier"]          # the ID before 3b/3c
all_works["correction"] = pd.Series(np.nan, index=all_works.index, dtype=object)

to_check_redirect = sorted((set(canon_list["GND"].dropna()) | set(all_works["gndIdentifier"].dropna())) - set(gnd_one.index))
redirect_results = {gid: check_id(gid) for gid in to_check_redirect}
save_cache("gnd_redirect_cache.json", redirect_cache)

redirects = {g: resolve(g) for g in to_check_redirect if resolve(g) != g}
unreachable = [g for g, r in redirect_results.items() if r is None]
print(len(to_check_redirect), "IDs checked,", len(redirects), "redirected,", len(unreachable), "not reachable")
if unreachable:
    print("lobid was not reachable for some IDs -> run this cell again (finished queries are cached)")
pd.Series({g: (r or {}).get("status") for g, r in redirect_results.items()}).value_counts()

403 IDs checked, 58 redirected, 0 not reachable


200    343
301     58
500      2
Name: count, dtype: int64

In [33]:
# canon list: old -> new ID
canon_list["GND_resolved"] = canon_list["GND"].map(lambda g: redirects.get(g, g))
canon_redirects = canon_list[canon_list["GND"] != canon_list["GND_resolved"]][["author", "GND", "GND_resolved", "canonicity_rank"]]
canon_dup = canon_list["GND_resolved"].notna() & canon_list["GND_resolved"].duplicated(keep=False)
if canon_dup.any():
    print("WARNING: after resolving, these persons are in the canon list twice:")
    display(canon_list[canon_dup])
print(len(canon_redirects), "outdated IDs in the canon list")
canon_redirects.sort_values("canonicity_rank")

58 outdated IDs in the canon list


,author,GND,GND_resolved,canonicity_rank
949,"König, Heinrich",1236637305,118724320,801
1021,"Burkart, Erika",1244000795,118517694,801
1300,"Neumark, Georg",1037528700,118587404,1293
2505,"Steigentesch, August von",118913840,117237361,1433
2424,"Tau, Max",1243928719,118620959,1433
2412,"Trenck, Friedrich, Freiherr von der",1243924470,118623796,1433
2282,"Eisendle, Helmut",172061024,12030550X,1433
2275,"Nicolai, Philipp",1243923822,11873475X,1433
2270,"Storz, Gerhard",1243891858,118618733,1433
2269,"Stern, Carola",1243891831,118809741,1433


In [34]:
# corpora: old -> new ID
def note(mask, text):
    all_works.loc[mask, "correction"] = all_works.loc[mask, "correction"].map(lambda c: text if pd.isna(c) else f"{c}; {text}")

moved = all_works["gndIdentifier"].isin(redirects)
all_works.loc[moved, "gndIdentifier"] = all_works.loc[moved, "gndIdentifier"].map(redirects)
note(moved, "GND redirect")
print(moved.sum(), "works with an outdated ID in the corpora")

0 works with an outdated ID in the corpora


## 3c Verified single fixes

**`ID_FIXES`** replaces an ID in **all** corpora (the ID is wrong for every work that has it).
**`ROW_FIXES`** changes single works (corpus + `filename`; for a work with several authors add the author position as a 6th element, e.g. `(..., "reason", 2)`), **`YEAR_FIXES`** the year of works picked by a condition.

Errors that come from the **RoKo metadata** are also noted in `RoKo_errata.md` in the project folder.
Every fix ends up in `corrections_log` and in the `correction` column of the export.

The fixes were checked one by one on 10.11.2026 (lobid.org / d-nb.info and the file names).
To add a fix: put it into the lists below and re-run from **Combine everything into one table**.

In [35]:
ID_FIXES = [
    # (old ID, new ID, reason)
    # the first two are already swapped in the 'exchange gndIdentifiers directly' cell in 3a -> no works left here
    ("1014122880", "118731092", "Marlitt: 1014122880 is a name record (not a person record), it comes from ELTeC "
                                "and was copied to other corpora in step 2b"),
    ("1014784034", "118582054", "Meysenbug: 1014784034 is a name record (ELTeC DEU086)"),
    ("116512288",  "118557211", "Jean Paul: 116512288 is the art historian Jean Paul Richter (1847-1937)"),
    ("127610524X", "118709933", "Henry von Heiseler: two GND records with identical life dates (1875-1928); "
                                "the canon list uses 118709933"),
    ("1050700996", "118521101", "H. Clauren (Carl Heun): second GND record for the same person; "
                                "the canon list and the rest of the corpora use 118521101"),
]

ROW_FIXES = [
    # (corpus, filename, column, new value, reason[, author_pos])   author_pos: only for works with several authors
    ("07_bildungsromane", "hoffmann_murr_t1.txt", "gndIdentifier", "118552465",
     "E.T.A. Hoffmann instead of Ernst Lothar Hoffmann (1898-1985)"),
    ("07_bildungsromane", "hoffmann_murr_t2.txt", "gndIdentifier", "118552465",
     "E.T.A. Hoffmann instead of Ernst Lothar Hoffmann (1898-1985)"),
    ("03_de-corp", "Coster,_Charles_de_-_Tyll_Ulenspiegel_und_Lamm_Goedzak:_Legende_von_ihren_heroischen,_lustigen_und_r-68862.txt",
     "gndIdentifier", "118677055", "Charles de Coster instead of Wilhelm Fischer (error in MANUAL_IDS)"),
    ("03_de-corp", "Karl_Grosse_-_Der_Genius.txt", "gndIdentifier", "120671883",
     "Carl Grosse (1768-1847); 116869852 is Karl Grosse (1804-1885)"),
    ("08_roko", "Grosse_Genius", "gndIdentifier", "120671883",
     "RoKo metadata: ID of Julius Grosse instead of Carl Grosse"),
    ("08_roko", "Grosse_Ravensbeck", "author_first", "Julius",
     "RoKo metadata: Ravensbeck (1910) is by Julius Grosse, not Carl Grosse (the ID was correct)"),
    ("05_ELTeC-deu", "DEU086", "author_last", "Meysenbug", "name missing in the ELTeC metadata"),
    ("05_ELTeC-deu", "DEU086", "author_first", "Malwida von", "name missing in the ELTeC metadata"),
]

YEAR_FIXES = [
    # (corpus, condition as a function of the table, new year, reason)
    ("01_german-fiction", lambda d: d["year"].eq(1017), np.nan,
     "Müller-Guttenbrunn: year 1017 is a typo in the file name of the source"),
]

log_rows = []

for old, new_id, why in ID_FIXES:
    m = all_works["gndIdentifier"].eq(old)
    if not m.any():
        print(f"ID_FIX {old} -> {new_id}: no works left (already replaced upstream)")
        continue
    all_works.loc[m, "gndIdentifier"] = new_id
    note(m, why)
    log_rows.append({"type": "ID_FIX", "old": old, "new": new_id, "works": int(m.sum()),
                     "corpora": ", ".join(sorted(all_works.loc[m, "corpus"].unique())), "reason": why})

for corpus, fn, col, val, why, *pos in ROW_FIXES:
    m = all_works["corpus"].eq(corpus) & all_works["filename"].eq(fn)
    if pos:
        m &= all_works["author_pos"].eq(pos[0])
    assert m.sum() == 1, f"not found (or not unique): {corpus} / {fn} (work with several authors? add the author position as 6th element)"
    old = all_works.loc[m, col].iloc[0]
    all_works.loc[m, col] = val
    note(m, why)
    log_rows.append({"type": f"ROW_FIX {col}", "old": old, "new": val, "works": 1, "corpora": corpus,
                     "file": fn, "reason": why})

for corpus, cond, val, why in YEAR_FIXES:
    m = all_works["corpus"].eq(corpus) & cond(all_works)
    old = ", ".join(map(str, all_works.loc[m, "year"].unique()))
    all_works.loc[m, "year"] = val
    note(m, why)
    log_rows.append({"type": "ROW_FIX year", "old": old, "new": val, "works": int(m.sum()), "corpora": corpus,
                     "file": "; ".join(all_works.loc[m, "filename"].astype(str)), "reason": why})

# every work whose ID changed (redirect or fix) is marked in the status column
changed = all_works["gndIdentifier"].fillna("") != all_works["gndIdentifier_orig"].fillna("")
all_works.loc[changed, "gnd_match_status"] = "corrected"
all_works["author_key"] = author_key(all_works["author_last"], all_works["author_first"])   # names may have changed

corrections_log = pd.DataFrame(log_rows)
print(changed.sum(), "works with a new ID,", all_works["correction"].notna().sum(), "works corrected in total")
corrections_log

74 works with a new ID, 77 works corrected in total


,type,old,new,works,corpora,reason,file
0,ID_FIX,1014122880,118731092,34,"01_german-fiction, 03_de-corp, 04_d-prose, 05_...",Marlitt: 1014122880 is a name record (not a pe...,NaN
1,ID_FIX,1014784034,118582054,1,05_ELTeC-deu,Meysenbug: 1014784034 is a name record (ELTeC ...,NaN
2,ID_FIX,116512288,118557211,23,"03_de-corp, 03_de-corp-non-fiction",Jean Paul: 116512288 is the art historian Jean...,NaN
3,ID_FIX,127610524X,118709933,1,04_d-prose,Henry von Heiseler: two GND records with ident...,NaN
4,ID_FIX,1050700996,118521101,11,"03_de-corp, 08_roko",H. Clauren (Carl Heun): second GND record for ...,NaN
5,ROW_FIX gndIdentifier,1111522545,118552465,1,07_bildungsromane,E.T.A. Hoffmann instead of Ernst Lothar Hoffma...,hoffmann_murr_t1.txt
6,ROW_FIX gndIdentifier,1111522545,118552465,1,07_bildungsromane,E.T.A. Hoffmann instead of Ernst Lothar Hoffma...,hoffmann_murr_t2.txt
7,ROW_FIX gndIdentifier,116583959,118677055,1,03_de-corp,Charles de Coster instead of Wilhelm Fischer (...,"Coster,_Charles_de_-_Tyll_Ulenspiegel_und_Lamm..."
8,ROW_FIX gndIdentifier,116869852,120671883,1,03_de-corp,Carl Grosse (1768-1847); 116869852 is Karl Gro...,Karl_Grosse_-_Der_Genius.txt
9,ROW_FIX gndIdentifier,120671883,120671883,1,08_roko,RoKo metadata: ID of Julius Grosse instead of ...,Grosse_Genius


# Step 4: add GND info + canonicity

Done once, with the final IDs from step 3: the GND name (IDs that are not in the big GND file get it from lobid) and the
canonicity scores, matched via the **resolved** IDs of the canon list (`GND_resolved`, step 3b).
`canonicity_orig` is the canonicity value a corpus ships itself.
At the end every work gets one **representative author** (`is_primary`): the author with the highest canonicity score; if no author has a score, the first author with an ID, otherwise the first author. Step 6 exports this author as the author of the work.

In [36]:
# GND name (and EXTRA_GND_COLS) from the GND file; IDs that are not in the file get the name from lobid
has_gnd = all_works["gndIdentifier"].notna()
for col in ["GND_name", *EXTRA_GND_COLS]:
    all_works[col] = all_works["gndIdentifier"].map(gnd_one[col])

def lobid_name(gid):
    rec = lobid_by_id(gid)
    return rec.get("preferredName") if rec and rec.get("found") else None

missing_name = has_gnd & all_works["GND_name"].isna()
all_works.loc[missing_name, "GND_name"] = all_works.loc[missing_name, "gndIdentifier"].map(lobid_name)
save_cache("lobid_id_cache.json", lobid_id_cache)

# canonicity score / rank of the canon list, matched via the resolved IDs
canon_by_id = canon_list.dropna(subset=["GND_resolved"]).drop_duplicates("GND_resolved").set_index("GND_resolved")
for col in ["canonicity_score", "canonicity_rank"]:
    all_works[col] = all_works["gndIdentifier"].map(canon_by_id[col])
all_works["canonicity_rank"] = all_works["canonicity_rank"].astype("Int64")

print("works with an ID but without GND_name:", int((has_gnd & all_works["GND_name"].isna()).sum()))
print("works with a canonicity score:", all_works["canonicity_score"].notna().sum())

# original canonicity measure that ships with a corpus' own metadata: corpus -> (raw metadata table, column)
# data_all['canonicity_orig'] gets the CONTENT of that column (e.g. ELTeC 'reprint-count': high / medium / ...).
# Add further corpora here.
ORIG_CANON = {
    "05_ELTeC-deu": (eltec_raw, "reprint-count"),
    "10_19C-prose": (prose_raw, "Kanon_Status"),
}
all_works["canonicity_orig"] = pd.Series(np.nan, index=all_works.index, dtype=object)
for corpus, (raw, col) in ORIG_CANON.items():
    idx = all_works.index[all_works["corpus"] == corpus]
    values = raw[col].reset_index(drop=True).iloc[all_works.loc[idx, "src_row"].to_numpy()].to_numpy()
    all_works.loc[idx, "canonicity_orig"] = pd.Series(values, index=idx).map(
        lambda v: np.nan if pd.isna(v) or str(v).strip() == "" else str(v).strip())
print("original canonicity values:")
print(all_works.groupby("corpus", sort=False)["canonicity_orig"].value_counts(dropna=False).loc[lambda s: s.index.get_level_values(0).isin(ORIG_CANON)])

# representative author of each work = the author with the highest canonicity score
# (ties / no score at all: the first author that has an ID, then the first author)
pick = pd.DataFrame({"work_id": all_works["work_id"],
                     "score": pd.to_numeric(all_works["canonicity_score"], errors="coerce").fillna(-np.inf),
                     "has_id": all_works["gndIdentifier"].notna(),
                     "pos": all_works["author_pos"]})
chosen = pick.sort_values(["work_id", "score", "has_id", "pos"], ascending=[True, False, False, True]).drop_duplicates("work_id")
all_works["is_primary"] = all_works.index.isin(chosen.index)
assert all_works["is_primary"].sum() == all_works["work_id"].nunique()
n_multi = all_works.loc[all_works["n_authors"] > 1, "work_id"].nunique()
print(f"{n_multi} works with several authors; the representative author is the one with the highest canonicity score")
# sanity check of the direction: a negative value means 'high score = low rank number = more canonical'
print("Spearman(score, rank) in the canon list:", round(canon_list[["canonicity_score", "canonicity_rank"]].corr(method="spearman").iloc[0, 1], 2))


works with an ID but without GND_name: 0
works with a canonicity score: 14489
original canonicity values:
corpus        canonicity_orig
10_19C-prose  0                  3463
              1                  1218
              2                   347
              3                   305
05_ELTeC-deu  high                 48
              low                  46
              unspecified           6
Name: count, dtype: int64
93 works with several authors; the representative author is the one with the highest canonicity score
Spearman(score, rank) in the canon list: -1.0


# Step 5: checks

Nothing is changed here, everything is checked on the final IDs. If something looks wrong, add a decision or a fix in
step 3 and re-run from **Combine everything into one table**.

### 5a How many authors are still open after each step?

In [37]:
# columns "after ...": author rows without ID that are still open (multi-author placeholders / anonymous / manual NoID not counted)
by_corpus = all_works.groupby("corpus", sort=False)
progress = pd.DataFrame(history)
progress.insert(0, "works", by_corpus["work_id"].nunique())
progress.insert(1, "author_rows", by_corpus.size())
progress.insert(2, "with_id", by_corpus["gndIdentifier"].count())
progress.loc["TOTAL"] = progress.sum()
progress["pct_with_id"] = (progress["with_id"] / progress["author_rows"] * 100).round(1)
progress.astype({c: int for c in progress.columns if c != "pct_with_id"})


,works,author_rows,with_id,1 metadata,2a string match,2b other corpora,2c lobid,3a manual,pct_with_id
corpus,,,,,,,,,
01_german-fiction,2735,2735,2735,2735,216,185,51,0,100.0
01_translated-fiction,484,484,484,484,484,484,154,0,100.0
03_de-corp,4215,4226,4208,4226,303,245,73,0,99.6
03_de-corp-non-fiction,765,771,760,768,246,244,47,0,98.6
04_d-prose,2511,2513,2513,2513,108,90,11,0,100.0
05_ELTeC-deu,100,100,100,0,0,0,0,0,100.0
06_romane-der-romantik,115,115,115,115,3,3,1,0,100.0
07_bildungsromane,220,222,222,222,18,16,5,0,100.0
08_roko,1834,1834,1834,1,0,0,0,0,100.0


In [38]:
# where the IDs come from (works)
pd.crosstab(all_works["corpus"], all_works["gnd_match_status"].fillna("-"), margins=True, margins_name="TOTAL")

gnd_match_status,anonymous,corrected,exact,exact+disambiguated,filled_from_other_corpus,filled_from_roko,first_given_name,first_given_name+disambiguated,lobid_match,manual_no_id,manual_override,multiple_authors,other_given_name,other_given_name+disambiguated,provided_in_metadata,wikidata,without_particles,without_titles,TOTAL
corpus,,,,,,,,,,,,,,,,,,,
01_german-fiction,0,11,2322,155,5,15,21,0,134,0,51,0,2,7,0,0,7,5,2735
01_translated-fiction,0,0,0,0,0,0,0,0,330,0,154,0,0,0,0,0,0,0,484
03_de-corp,0,42,3493,245,5,43,44,2,171,18,55,0,11,1,0,0,30,66,4226
03_de-corp-non-fiction,2,1,441,21,2,0,35,6,197,8,40,1,2,0,0,0,9,6,771
04_d-prose,0,8,2181,193,2,9,18,0,79,0,11,0,0,0,0,0,12,0,2513
05_ELTeC-deu,0,4,0,0,0,0,0,0,0,0,6,0,0,0,90,0,0,0,100
06_romane-der-romantik,0,0,82,10,0,0,9,0,2,0,1,0,4,1,0,0,6,0,115
07_bildungsromane,0,2,196,1,0,2,5,0,11,0,5,0,0,0,0,0,0,0,222
08_roko,0,3,1,0,0,0,0,0,0,0,6,0,0,0,1824,0,0,0,1834


### 5b Authors with different IDs (same name, several IDs)

In [39]:
roko_pairs = set(zip(all_works.loc[all_works["corpus"] == PREFERRED_CORPUS, "author_key"],
                     all_works.loc[all_works["corpus"] == PREFERRED_CORPUS, "gndIdentifier"]))
ids_now = all_works.dropna(subset=["gndIdentifier"]).groupby("author_key")["gndIdentifier"].nunique()
conflicts = (all_works[all_works["author_key"].isin(ids_now[ids_now > 1].index) & all_works["gndIdentifier"].notna()]
             .groupby(["author_key", "gndIdentifier"])
             .agg(works=("corpus", "size"), corpora=("corpus", lambda s: ", ".join(sorted(s.unique()))),
                  status=("gnd_match_status", lambda s: ", ".join(sorted(s.dropna().unique()))))
             .reset_index()
             .assign(in_roko=lambda d: [(k, g) in roko_pairs for k, g in zip(d["author_key"], d["gndIdentifier"])])
             .merge(gnd_one[["GND_name", "GND_birth", "GND_death", "GND_occupation"]], left_on="gndIdentifier",
                    right_index=True, how="left"))
print(conflicts["author_key"].nunique(), "authors with more than one ID")
conflicts

1 authors with more than one ID


,author_key,gndIdentifier,works,corpora,status,in_roko,GND_name,GND_birth,GND_death,GND_occupation
0,"grosse, julius",118718983,3,"03_de-corp, 10_19C-prose, 13_deutscher-novelle...","exact, provided_in_metadata",False,"Grosse, Julius",1828.0,1902.0,"Schriftsteller, Redakteur, Theaterkritiker, Ly..."
1,"grosse, julius",120671883,1,08_roko,manual_override,True,"Grosse, Carl",1768.0,1847.0,"Schriftsteller, Mineraloge, Geologe"


In [40]:
all_works.query("author_last=='Zschokke'")

,author_first,author_last,title,year,filename,gndIdentifier,gnd_match_status,wikidata_id,corpus,src_row,...,author_key,stringmatch_id,stringmatch_status,gndIdentifier_orig,correction,GND_name,canonicity_score,canonicity_rank,canonicity_orig,is_primary
1129,Heinrich,Zschokke,Addrich im Moos,1825.0,Heinrich_Zschokke_-_Addrich_im_Moos_(1825).txt,118637266,exact,NaN,01_german-fiction,1129,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1130,Heinrich,Zschokke,Alamontade,1803.0,Heinrich_Zschokke_-_Alamontade_(1803).txt,118637266,exact,NaN,01_german-fiction,1130,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1131,Heinrich,Zschokke,Das Goldmacherdorf,1817.0,Heinrich_Zschokke_-_Das_Goldmacherdorf_(1817).txt,118637266,exact,NaN,01_german-fiction,1131,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1132,Heinrich,Zschokke,Das blaue Wunder,1829.0,Heinrich_Zschokke_-_Das_blaue_Wunder_(1829).txt,118637266,exact,NaN,01_german-fiction,1132,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1133,Heinrich,Zschokke,Der Freihof von Aarau,1823.0,Heinrich_Zschokke_-_Der_Freihof_von_Aarau_(182...,118637266,exact,NaN,01_german-fiction,1133,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1134,Heinrich,Zschokke,Der König von Akim,1825.0,Heinrich_Zschokke_-_Der_König_von_Akim_(1825)...,118637266,exact,NaN,01_german-fiction,1134,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1135,Heinrich,Zschokke,Der Pascha von Buda,1819.0,Heinrich_Zschokke_-_Der_Pascha_von_Buda_(1819)...,118637266,exact,NaN,01_german-fiction,1135,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1136,Heinrich,Zschokke,Die Branntweinpest,1837.0,Heinrich_Zschokke_-_Die_Branntweinpest_(1837).txt,118637266,exact,NaN,01_german-fiction,1136,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1137,Heinrich,Zschokke,Die Rose von Disentis,1822.0,Heinrich_Zschokke_-_Die_Rose_von_Disentis_(182...,118637266,exact,NaN,01_german-fiction,1137,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True
1138,Heinrich,Zschokke,Jonathan Frock,1825.0,Heinrich_Zschokke_-_Jonathan_Frock_(1825).txt,118637266,exact,NaN,01_german-fiction,1138,...,"zschokke, heinrich",118637266,exact,118637266,NaN,"Zschokke, Heinrich",3.814391,427,NaN,True


### 5c One ID for authors with different last names

Fine for pseudonyms (H. Clauren = Karl Heun), otherwise probably a copy-paste error.

In [41]:
checked = ["100801919", "101198566", "106115200", "106537970", "107514176", "113665946", "115341080", "115341307",
           "116001089", "116045469", "116108118", "116137312", "116302712", "116363207", "11642284X", "116457139",
           "116563559", "116565616", ]

w = all_works.dropna(subset=["gndIdentifier"]).assign(last_norm=lambda d: d["author_last"].map(norm))
shared = w.groupby("gndIdentifier")["last_norm"].nunique()
shared_ids = (w[w["gndIdentifier"].isin(shared[shared > 1].index)]
              .groupby(["gndIdentifier", "author_key"])
              .agg(works=("corpus", "size"), status=("gnd_match_status", "first"))
              .reset_index()
              .merge(gnd_one[["GND_name"]], left_on="gndIdentifier", right_index=True, how="left"))
print(shared_ids.query("gndIdentifier.isin(@checked)==False")["gndIdentifier"].nunique(), "Unchecked IDs used for differently named authors")
shared_ids.query("gndIdentifier.isin(@checked)==False").head(15)

66 Unchecked IDs used for differently named authors


,gndIdentifier,author_key,works,status,GND_name
43,116650818,"heigel, karl von",3,exact,"Heigel, Karl von"
44,116650818,"karl heigel, august",6,manual_override,"Heigel, Karl von"
45,116650818,"karl heigel, august von",6,manual_override,"Heigel, Karl von"
46,116650818,"karl heigel, v",17,manual_override,"Heigel, Karl von"
47,116679913,"mahn, anny",2,provided_in_metadata,"Mahn, Anny"
48,116679913,"wothe, anny",2,lobid_match,"Mahn, Anny"
49,116682477,"gluemer, claire von",4,exact,"Glümer, Claire von"
50,116682477,"von gluemer, claire",1,provided_in_metadata,"Glümer, Claire von"
51,11678699X,"heyden, friedrich von",1,exact,"Heyden, Friedrich von"
52,11678699X,"heyden, friedrich von der",1,without_particles,"Heyden, Friedrich von"


In [42]:
shared_ids.query("gndIdentifier=='113665946'")

,gndIdentifier,author_key,works,status,GND_name
14,113665946,"auer, adelheid von",4,lobid_match,"Cosel, Charlotte von"
15,113665946,"cosel, charlotte von",4,provided_in_metadata,"Cosel, Charlotte von"


### 5d IDs from the metadata / Wikidata vs. the string match

In [43]:
provided = all_works["gnd_match_status"].isin(["provided_in_metadata", "wikidata", "wikidata+disambiguated"])
disagree = provided & all_works["stringmatch_id"].notna() & (all_works["stringmatch_id"] != all_works["gndIdentifier"])
print(disagree.sum(), "works where the name lookup finds a different ID than the metadata")
(all_works[disagree]
 .drop_duplicates(["author_key", "gndIdentifier", "stringmatch_id"])
 [["corpus", "author_first", "author_last", "gndIdentifier", "stringmatch_id", "stringmatch_status"]]
 .assign(provided_name=lambda d: d["gndIdentifier"].map(gnd_one["GND_name"]),
         lookup_name=lambda d: d["stringmatch_id"].map(gnd_one["GND_name"])))

25 works where the name lookup finds a different ID than the metadata


,corpus,author_first,author_last,gndIdentifier,stringmatch_id,stringmatch_status,provided_name,lookup_name
13017,09_ecocor,Theodor,Hoffmann,118552465,116941731,exact,"Hoffmann, E. T. A.","Hoffmann, Theodor"
13022,09_ecocor,Ernst,Hoffmann,118552465,1111522545,exact,"Hoffmann, E. T. A.","Hoffmann, Ernst Lothar"
13168,09_ecocor,Alfred,Henschke,118562681,1182379923,exact,Klabund,"Henschke, Alfred"
18612,11_dta-belletristik,Johann Justus,Ebeling,101240937,124799914,first_given_name,NaN,"Ebeling, Johann"
18907,11_dta-belletristik,Franz von,Pocci,118595245,102260664,exact,"Pocci, Franz, Graf von","Pocci, Franz von"
19179,13_deutscher-novellenschatz,Bernhard,Abeken,116001089,116001097,exact+disambiguated,"Abeken, Bernhard","Abeken, Bernhard Rudolf"
19224,13_deutscher-novellenschatz,Hermann,Kurz,118778277,12460272X,exact+disambiguated,"Kurz, Hermann","Kurz, Hermann"
19227,13_deutscher-novellenschatz,Heinrich,Landesmann,11872911X,131813904X,exact,"Lorm, Hieronymus","Landesmann, Heinrich"
19245,13_deutscher-novellenschatz,Levin,Schücking,118762109,117124931,exact+disambiguated,"Schücking, Levin","Schücking, Levin Ludwig"
19246,13_deutscher-novellenschatz,Karl,Spindler,119164043,139940359,exact+disambiguated,"Spindler, Karl","Spindler, Karl"


In [44]:
all_works.query("author_last=='Henschke' and corpus=='09_ecocor'")

,author_first,author_last,title,year,filename,gndIdentifier,gnd_match_status,wikidata_id,corpus,src_row,...,author_key,stringmatch_id,stringmatch_status,gndIdentifier_orig,correction,GND_name,canonicity_score,canonicity_rank,canonicity_orig,is_primary
13168,Alfred,Henschke,Moreau,1915.0,1915_Klabund_Moreau.xml,118562681,wikidata,Q61263,09_ecocor,168,...,"henschke, alfred",1182379923,exact,118562681,NaN,Klabund,2.331067,801,NaN,True
13174,Alfred,Henschke,Mohammed,1917.0,1917_Klabund_Mohammed.xml,118562681,wikidata,Q61263,09_ecocor,174,...,"henschke, alfred",1182379923,exact,118562681,NaN,Klabund,2.331067,801,NaN,True
13187,Alfred,Henschke,Franziskus,1921.0,1921_Klabund_Franziskus.xml,118562681,wikidata,Q61263,09_ecocor,187,...,"henschke, alfred",1182379923,exact,118562681,NaN,Klabund,2.331067,801,NaN,True
13188,Alfred,Henschke,Hieronymus,1928.0,1928_Klabund_Hieronymus.xml,118562681,wikidata,Q61263,09_ecocor,188,...,"henschke, alfred",1182379923,exact,118562681,NaN,Klabund,2.331067,801,NaN,True


### 5e Do all IDs exist?

IDs that are not in the GND file are checked on lobid (cached). `not found` = probably a typo;
`redirected_to` = the GND merged this record into another one (step 3b already resolves these, so nothing should be left); `not a person` = wrong record type.

In [45]:
ids = pd.Series(all_works["gndIdentifier"].dropna().unique())
bad_format = ids[~ids.str.match(ID_RE)]
to_check = ids[ids.str.match(ID_RE) & ~ids.isin(gnd_one.index)]
print(len(ids), "IDs in use,", len(bad_format), "with a wrong format,", len(to_check), "not in the GND file -> lobid")

checked = []
for gid in to_check:
    rec = lobid_by_id(gid)
    if rec is None:
        problem = "lobid not reachable (run again)"
    elif not rec["found"]:
        problem = "not found"
    elif rec.get("gndIdentifier") and rec["gndIdentifier"] != gid:
        problem = f"redirected_to {rec['gndIdentifier']}"
    elif "Person" not in (rec.get("type") or []):
        problem = "not a person"
    else:
        continue
    checked.append({"gndIdentifier": gid, "problem": problem})
save_cache("lobid_id_cache.json", lobid_id_cache)

id_problems = pd.DataFrame([{"gndIdentifier": g, "problem": "wrong format"} for g in bad_format] + checked,
                           columns=["gndIdentifier", "problem"])
id_problems = id_problems.merge(
    all_works.groupby("gndIdentifier").agg(authors=("author_key", lambda s: "; ".join(sorted(s.unique()))),
                                           works=("corpus", "size"), status=("gnd_match_status", "first")),
    left_on="gndIdentifier", right_index=True, how="left")
print(len(id_problems), "IDs with a problem")
id_problems

1836 IDs in use, 0 with a wrong format, 342 not in the GND file -> lobid
0 IDs with a problem


,gndIdentifier,problem,authors,works,status


### 5f Review list: same last name + initial, several IDs

Nothing is changed here. The lists show candidates for further errors; whatever turns out to be one goes into
`ID_FIXES` / `ROW_FIXES` (3c). Both lists are exported.

Same last name and first initial, but several IDs, at least one of them without a canonicity score.
Catches cases like Marlitt (the same person as 'E.' and 'Eugenie'), which the check in 5b does not see.
Most hits are really different people (e.g. Hans and Heinrich Hart).

In [46]:
cw = all_works[all_works["gndIdentifier"].notna() & all_works["corpus"].ne("01_translated-fiction")].copy()
cw["ln"], cw["fi"] = cw["author_last"].map(norm), cw["author_first"].map(norm).str[:1]
grp = cw.groupby(["ln", "fi"])
flag = grp["gndIdentifier"].transform("nunique").gt(1) & grp["canonicity_score"].transform(lambda s: s.isna().any())
check_split = (cw[flag].groupby(["ln", "fi", "gndIdentifier", "GND_name"], dropna=False)
               .agg(names=("author_first", lambda s: " | ".join(sorted(s.dropna().astype(str).unique()))),
                    works=("corpus", "size"), canon_rank=("canonicity_rank", "first"),
                    corpora=("corpus", lambda s: ", ".join(sorted(s.unique()))))
               .reset_index())
print(check_split[["ln", "fi"]].drop_duplicates().shape[0], "name groups to look at")
check_split

15 name groups to look at


,ln,fi,gndIdentifier,GND_name,names,works,canon_rank,corpora
0,becker,k,108972933,"Becker, Kurt E.",Kurt E.,1,<NA>,07_bildungsromane
1,becker,k,119090716,"Becker, Karl Friedrich",Karl Friedrich,3,<NA>,03_de-corp
2,bley,f,116201878,"Bley, Fritz",Fritz,3,<NA>,"03_de-corp, 03_de-corp-non-fiction"
3,bley,f,118511653,"Blei, Franz",Franz,2,542,03_de-corp-non-fiction
4,grimm,h,118542249,"Grimm, Hans",Hans,1,370,07_bildungsromane
5,grimm,h,118697773,"Grimm, Herman",Herman,1,801,13_deutscher-novellenschatz
6,grimm,h,124670245,"Grimm, Hermann",Hermann,1,<NA>,10_19C-prose
7,hahn,j,1018778209,"Hahn, Julius",Julius,1,<NA>,03_de-corp-non-fiction
8,hahn,j,11638669X,"Hahn, Johann Friedrich",Johann,2,1433,"01_german-fiction, 04_d-prose"
9,hart,h,117122440,"Hart, Hans",Hans | Hans,8,<NA>,"01_german-fiction, 03_de-corp, 04_d-prose, 08_..."


### 5g Review list: works without a score whose author is in the canon list under another ID

Works without a canonicity score whose author is in the canon list (same last name and first initial) - with a different ID.
Catches cases like Jean Paul or E.T.A. Hoffmann (wrong person with the same name) and remaining duplicates.
Column `same_name`: first and last name are identical - these are the most important candidates.

In [47]:
cn = (canon_list.assign(ln=canon_list["author"].str.split(",").str[0].map(norm),
                        canon_first=canon_list["author"].str.split(",").str[1].map(norm))
      .rename(columns={"author": "canon_name", "GND_resolved": "canon_GND", "canonicity_rank": "canon_rank"}))
cn["fi"] = cn["canon_first"].str[:1]
unscored = cw[cw["canonicity_score"].isna()]
check_canon = (unscored.merge(cn[["ln", "fi", "canon_first", "canon_GND", "canon_name", "canon_rank"]], on=["ln", "fi"])
               .assign(same_name=lambda d: d["author_first"].map(norm).eq(d["canon_first"]))
               .groupby(["author_first", "author_last", "gndIdentifier", "GND_name", "canon_GND", "canon_name",
                         "canon_rank", "same_name"], dropna=False)
               .agg(works=("corpus", "size"), corpora=("corpus", lambda s: ", ".join(sorted(s.unique()))))
               .reset_index()
               .sort_values(["same_name", "works"], ascending=False))
check_canon

,author_first,author_last,gndIdentifier,GND_name,canon_GND,canon_name,canon_rank,same_name,works,corpora
10,Georg,Schwarz,173675700,"Schwarz, Georg",117366714,"Schwarz, Georg",2670,True,1,03_de-corp-non-fiction
38,Heinrich,Seidel,118796089,"Seidel, Heinrich",118760602,"Seidel, Heinrich Wolfgang",3817,False,68,"01_german-fiction, 04_d-prose, 08_roko, 09_ecocor"
26,Alfred,Schirokauer,117276464,"Schirokauer, Alfred",118795066,"Schirokauer, Arno",1433,False,20,"01_german-fiction, 04_d-prose, 08_roko"
31,Emma,Meier,1415405654,"Meier, Emma",118580043,"Meier, Emerenz",801,False,19,10_19C-prose
13,Heinrich,Seidel,118796089,"Seidel, Heinrich",118760602,"Seidel, Heinrich Wolfgang",3817,False,14,03_de-corp
0,Alfred,Schirokauer,117276464,"Schirokauer, Alfred",118795066,"Schirokauer, Arno",1433,False,11,03_de-corp
3,Arthur,Schopenhauer,118610465,"Schopenhauer, Arthur",11880510X,"Schopenhauer, Adele",801,False,8,03_de-corp-non-fiction
32,Ferdinand,Emmerich,127581413,"Emmerich, Ferdinand",118812882,"Emmerich, Friedrich Joseph",4234,False,7,01_german-fiction
33,Franz Wilhelm,Ziegler,116987286,"Ziegler, Franz Wilhelm",116987367,"Ziegler, Friedrich W.",3095,False,6,"10_19C-prose, 13_deutscher-novellenschatz"
35,Friedrich,Schulz,100316875,"Schulz, Friedrich",123749212,"Schulz, Frank",801,False,6,11_dta-belletristik


### 5h Works with several authors

Nothing is changed here. `multi_authors` shows every author of every split work (the representative author has `is_primary`);
`unsplit_multi` lists works that still have several authors in one row (the string match or the metadata found several names, but nobody split them).
Add them to `AUTHOR_SPLITS` (*Combine everything*) and re-run from there. Placeholders like 'Autoren, Verschiedene' and anonymous works are meant to stay.

In [48]:
m_rows = all_works[all_works["n_authors"] > 1]
print(m_rows["work_id"].nunique(), "works with several authors,", len(m_rows), "author rows")
multi_authors = (m_rows.sort_values(["work_id", "author_pos"])
                 [["corpus", "title", "author_pos", "author_role", "author_first", "author_last", "gndIdentifier", "GND_name",
                   "gnd_match_status", "canonicity_score", "is_primary"]])
display(multi_authors)

unsplit_multi = (all_works[all_works["gnd_match_status"].eq("multiple_authors") & all_works["n_authors"].eq(1)
                           & all_works["author_key"].ne("autoren, verschiedene")]
                 .drop_duplicates(["corpus", "author_last"])[["corpus", "title", "author_first", "author_last"]])
print(len(unsplit_multi), "works still have several authors in one row (-> AUTHOR_SPLITS)")
unsplit_multi

93 works with several authors, 186 author rows


,corpus,title,author_pos,author_role,author_first,author_last,gndIdentifier,GND_name,gnd_match_status,canonicity_score,is_primary
7654,03_de-corp-non-fiction,Das malerische und romantische Westphalen,1,author,Levin,Schücking,118762109,"Schücking, Levin",exact+disambiguated,2.331067,False
7655,03_de-corp-non-fiction,Das malerische und romantische Westphalen,2,author,Ferdinand,Freiligrath,118535196,"Freiligrath, Ferdinand",exact,8.269016,True
7841,03_de-corp-non-fiction,Wanderungen an der Nord- und Ostsee,1,author,Wilhelm,Cornelius,1341917541,"Cornelius, Wilhelm",manual_override,NaN,True
7842,03_de-corp-non-fiction,Wanderungen an der Nord- und Ostsee,2,author,Theodor von,Kobbe,119068133,"Kobbe, Theodor von",exact,NaN,False
7511,03_de-corp-non-fiction,Brehms Tierleben. Band 27.Ergänzungsband 3: Di...,1,author,Ernst Ludwig,Taschenberg,117614467,"Taschenberg, Ernst Ludwig",lobid_match,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...
19023,11_dta-belletristik,Gedichte,2,author,Friedrich Leopold zu,Stolberg-Stolberg,118755552,"Stolberg-Stolberg, Friedrich Leopold zu",provided_in_metadata,4.119892,True
19053,11_dta-belletristik,Herzensergießungen eines kunstliebenden Kloste...,1,author,Wilhelm Heinrich,Wackenroder,118628097,"Wackenroder, Wilhelm Heinrich",provided_in_metadata,17.342033,False
19054,11_dta-belletristik,Herzensergießungen eines kunstliebenden Kloste...,2,author,Ludwig,Tieck,12989432X,"Tieck, Ludwig",provided_in_metadata,159.781633,True
19096,12_diskoclassics,Leben Eduards des Zweiten von England,1,author,Bertolt,Brecht,118514768,"Brecht, Bertolt",exact,413.814569,True


0 works still have several authors in one row (-> AUTHOR_SPLITS)


,corpus,title,author_first,author_last


### 5i Summary

In [49]:
summary = {
    "authors still open (3a list)": len(open_authors),
    "works with several authors, still one row (5h)": len(unsplit_multi),
    "authors with several IDs (5b)": conflicts["author_key"].nunique(),
    "IDs shared by different last names (5c)": shared_ids["gndIdentifier"].nunique(),
    "metadata vs. string match (5d)": int(disagree.sum()),
    "IDs with a problem (5e)": len(id_problems),
    "review list: same name, several IDs (5f)": check_split[["ln", "fi"]].drop_duplicates().shape[0],
    "review list: canon list has another ID (5g)": len(check_canon),
}
for k, v in summary.items():
    print(f"{'OK ' if v == 0 else '-> '} {k}: {v}")

OK  authors still open (3a list): 0
OK  works with several authors, still one row (5h): 0
->  authors with several IDs (5b): 1
->  IDs shared by different last names (5c): 84
->  metadata vs. string match (5d): 25
OK  IDs with a problem (5e): 0
->  review list: same name, several IDs (5f): 15
->  review list: canon list has another ID (5g): 52


# Step 6: export

- `data_all.csv`: **one row per work**, all corpora, standardized columns (+ `title`, + `canonicity_orig`, the canonicity value a corpus ships itself, e.g. ELTeC `reprint-count`, 19C prose `Kanon_Status`;
  + `gndIdentifier_orig`, the ID before step 3b/3c, and `correction`, the reason where step 3 changed something).
  For a work with several authors the columns `author_first`, `author_last`, `gndIdentifier`, `GND_name`, `gnd_match_status` and the canonicity columns
  belong to the **representative author = the author with the highest canonicity score** (step 4). The new last columns describe all authors:
  `work_id`, `n_authors`, `authors_all` (`Last, First; Last, First`) and `gndIdentifier_all` (`ID; ID`, in the same order, `NoID` for missing IDs).
- `work_authors.csv`: **one row per (work, author)**, linked to the other files via `work_id`; with `author_pos`, `author_role`, `is_primary`, the ID, the GND name and the canonicity of each author.
- one CSV per corpus: **all original columns** of the corpus (its own metadata file where there is one, otherwise what
  could be read from file names / TEI headers) followed by the columns added by this notebook
  (`gndIdentifier`, `GND_name`, `gnd_match_status`, canonicity, `gndIdentifier_orig`, `correction`, `work_id`, `n_authors`, `authors_all`, `gndIdentifier_all`).
  Works without an ID get `NoID`. One row per work, in the order of the original file.
  For corpora with their own metadata file the original columns stay exactly as shipped, so a corrected name or year
  shows up in `data_all.csv` (and is explained in `correction`), not in the original columns.
- from step 3: `corrections_log.csv`, `canon_id_redirects.csv`; from step 5: the two review lists
  (`review_5f_same_name_several_ids.csv`, `review_5g_canon_list_other_id.csv`).


In [50]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- one row per work: the representative author (is_primary = highest canonicity score, step 4) + all authors ----------
aw = all_works.assign(
    _name=(all_works["author_last"].fillna("") + ", " + all_works["author_first"].fillna("")).str.strip(", "),
    _gid=all_works["gndIdentifier"].fillna("NoID"),
).sort_values(["work_id", "author_pos"])
per_work = aw.groupby("work_id").agg(authors_all=("_name", "; ".join), gndIdentifier_all=("_gid", "; ".join))
works_out = (all_works[all_works["is_primary"]]                       # same order as the corpora (corpus, src_row)
             .merge(per_work, left_on="work_id", right_index=True, how="left")
             .reset_index(drop=True))
assert len(works_out) == all_works["work_id"].nunique()
assert works_out.groupby("corpus", sort=False)["src_row"].apply(lambda s: s.is_monotonic_increasing).all()   # same order as the corpora

# ---------- data_all.csv: standardized columns for all corpora, one row per work ----------
FINAL_COLS = ["corpus", "author_first", "author_last", "filename", "title", "year", "gndIdentifier", "GND_name",
              "gnd_match_status", *EXTRA_GND_COLS, "canonicity_score", "canonicity_rank", "canonicity_orig",
              "gndIdentifier_orig", "correction", "work_id", "n_authors", "authors_all", "gndIdentifier_all"]
final = works_out[FINAL_COLS].copy()

# title: from the metadata (step 1); where a corpus has none, infer it from 'Author_-_Title_(year).txt'-style file names
no_title = final["title"].isna() & final["filename"].notna()        # filename is NaN for a few works -> skipped
inferred = {}
for i, f in final.loc[no_title, "filename"].items():
    m = FILENAME_RE.match(str(f))
    if m:
        inferred[i] = m["title"].replace("_", " ").strip()
if inferred:                                                         # assign only real strings (str dtype rejects float NaN)
    final.loc[list(inferred), "title"] = list(inferred.values())
missing_title = final.loc[final["title"].isna()].groupby("corpus").size()
print("works without a title:", "none" if missing_title.empty else "\n" + missing_title.to_string())
final["year"] = final["year"].astype("Int64")                    # 1837, not 1837.0
final["gndIdentifier"] = final["gndIdentifier"].fillna("NoID")
final["gndIdentifier_orig"] = final["gndIdentifier_orig"].fillna("NoID")
final.to_csv(OUT_DIR / "data_all.csv", index=False, encoding="utf-8")
print(f"{'data_all':<25} {len(final):>5} works,", "NoID count : ", final.query("gndIdentifier=='NoID'").shape[0])

# ---------- one CSV per corpus: original columns + what this notebook added ----------
ORIGINAL = {                      # raw metadata tables exactly as shipped (same row order as the step 1 tables)
    "04_d-prose": d_prose_raw,
    "05_ELTeC-deu": eltec_raw,
    "06_romane-der-romantik": roro_raw,
    "07_bildungsromane": bild_raw,
    "08_roko": roko_raw,
    "10_19C-prose": prose_raw,
    "12_diskoclassics": disko_raw,
    "14_neuer-deutscher-novellenschatz": neuer_raw,
}                                 # every other corpus has no metadata file: its step 1 table is all there is
ADDED_COLS = ["gndIdentifier", "GND_name", "gnd_match_status", *EXTRA_GND_COLS, "canonicity_score", "canonicity_rank",
              "gndIdentifier_orig", "correction", "work_id", "n_authors", "authors_all", "gndIdentifier_all"]

for corpus, d in works_out.groupby("corpus", sort=False):
    orig = ORIGINAL.get(corpus, corpora[corpus]).reset_index(drop=True).copy()
    assert len(orig) == len(d), f"{corpus}: {len(orig)} original rows vs {len(d)} works"
    if corpus not in ORIGINAL:    # these two were created / filled by this notebook, the final values come from `works_out`
        orig = orig.drop(columns=["gndIdentifier", "gnd_match_status", "authors"], errors="ignore")   # "authors" = list column of step 1
        for col in ["author_first", "author_last", "year"]:        # names / years corrected in step 3c
            if col in orig:
                orig[col] = d.set_index("src_row")[col].reindex(range(len(orig))).to_numpy()
    orig = orig.rename(columns={c: f"{c}_orig" for c in orig.columns if c in ADDED_COLS})   # avoid name clashes
    if "year" in orig and pd.api.types.is_float_dtype(orig["year"]) and (orig["year"].dropna() % 1 == 0).all():
        orig["year"] = orig["year"].astype("Int64")                                          # 1837, not 1837.0

    added = d.set_index("src_row")[ADDED_COLS].reindex(range(len(orig))).reset_index(drop=True)
    added["gndIdentifier"] = added["gndIdentifier"].fillna("NoID")
    added["gndIdentifier_orig"] = added["gndIdentifier_orig"].fillna("NoID")
    out = pd.concat([orig, added], axis=1)
    out.to_csv(OUT_DIR / f"{corpus}.csv", index=False, encoding="utf-8")
    print(f"{corpus:<25} {len(out):>5} works, {out.shape[1]:>3} columns, {(added['gndIdentifier'] != 'NoID').mean():6.1%} with ID")

# ---------- work_authors.csv: one row per (work, author) ----------
AUTHOR_OUT_COLS = ["work_id", "corpus", "filename", "title", "year", "author_pos", "n_authors", "author_role", "is_primary",
                   "author_first", "author_last", "gndIdentifier", "GND_name", "gnd_match_status", *EXTRA_GND_COLS,
                   "canonicity_score", "canonicity_rank", "gndIdentifier_orig", "correction"]
authors_out = all_works[AUTHOR_OUT_COLS].copy()
authors_out["year"] = authors_out["year"].astype("Int64")
authors_out["gndIdentifier"] = authors_out["gndIdentifier"].fillna("NoID")
authors_out["gndIdentifier_orig"] = authors_out["gndIdentifier_orig"].fillna("NoID")
authors_out.to_csv(OUT_DIR / "work_authors.csv", index=False, encoding="utf-8")
print(f"{'work_authors':<25} {len(authors_out):>5} author rows for {authors_out['work_id'].nunique()} works,",
      authors_out.loc[authors_out['n_authors'] > 1, 'work_id'].nunique(), "with several authors")

# ---------- extras from step 3 and step 5 ----------
corrections_log.to_csv(OUT_DIR / "corrections_log.csv", index=False, encoding="utf-8")
canon_redirects.to_csv(OUT_DIR / "canon_id_redirects.csv", index=False, encoding="utf-8")
check_split.to_csv(OUT_DIR / "review_5f_same_name_several_ids.csv", index=False, encoding="utf-8")
check_canon.to_csv(OUT_DIR / "review_5g_canon_list_other_id.csv", index=False, encoding="utf-8")
print(int(all_works["correction"].notna().sum()), "works corrected in step 3")
print("->", OUT_DIR)

works without a title: 
corpus
03_de-corp      1
10_19C-prose    1
data_all                  19242 works, NoID count :  362
01_german-fiction          2735 works,  16 columns, 100.0% with ID
01_translated-fiction       484 works,  16 columns, 100.0% with ID
03_de-corp                 4215 works,  21 columns,  99.6% with ID
03_de-corp-non-fiction      765 works,  20 columns,  98.6% with ID
04_d-prose                 2511 works,  27 columns, 100.0% with ID
05_ELTeC-deu                100 works,  33 columns, 100.0% with ID
06_romane-der-romantik      115 works,  26 columns, 100.0% with ID
07_bildungsromane           220 works,  25 columns, 100.0% with ID
08_roko                    1834 works,  30 columns, 100.0% with ID
09_ecocor                   190 works,  17 columns, 100.0% with ID
10_19C-prose               5277 works,  49 columns,  93.8% with ID
11_dta-belletristik         553 works,  17 columns,  99.3% with ID
12_diskoclassics             86 works,  16 columns,  97.7% with ID
13_de